<a href="https://colab.research.google.com/github/Maverick-Ansh/delta_reasoner/blob/main/scratchpad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ∇-Reasoner, from scratch

### Test-time gradient descent in latent space

Rebuilding **∇-Reasoner: LLM Reasoning via Test-Time Gradient Descent in Latent Space** (Wang, Cai, Wang, Mei, Liu, Li, Wang — ICLR 2026, arXiv:2603.04948) piece by piece, on 2×T4.

---

## The one-sentence idea

Every other test-time scaling method is **zeroth-order**. Best-of-N, self-consistency, Tree-of-Thought, RAP: they all sample a bunch of candidate answers, score them, and keep the best. They only ever look at reward *values*.

But the reward model is a neural network. It is differentiable. So you can ask it a strictly richer question:

> not "is this answer good?" but **"which direction should I move this answer to make it better?"**

That is a **first-order** method. ∇-Reasoner takes the gradient of the reward with respect to the text, and does gradient descent on the text at inference time. No weights are updated. Nothing is trained. The descent happens inside a single forward pass of decoding and is thrown away afterwards.

## What "latent space" means here, precisely

This is the part that is easy to get wrong, so it is Rung 0 of this notebook.

There is a whole literature on "latent reasoning" that works on **hidden states** — Coconut, Huginn/recurrent-depth, pondering tokens. Those methods reach inside the model and iterate on `h`, the residual stream.

**∇-Reasoner does not do that.** It never touches the internals. It works on the **pre-softmax logits of the output tokens**, $z \in \mathbb{R}^{|y| \times |\mathcal{V}|}$. That is the continuous relaxation of the *discrete token sequence*, not of the model's internal state. The paper is explicit about this (App. A, "Continuous Latent Space Reasoning"):

> *"They operate within the LLM's latent space, modifying internal hidden representations. In contrast, our work, ∇-Reasoner, performs optimization directly in the output space."*

So the object being optimized is a **relaxed piece of text**. We will make that concrete and plottable in Rung 0.

---

## Claims we will try to falsify

Stated so they can come out false. Marked **H** for headline (an abstract number) or **M** for mechanism.

| # | Claim | Where | What would confirm it |
|---|---|---|---|
| **C1** | **M** Gradient flows *bidirectionally* along the sequence. Later tokens send signal back to earlier ones. Prior gradient-decoding work detaches this. | Prop. C.1, Remark C.2 | $\delta_{\text{postfix}} \neq 0$, and matching autograd to ~1e-6 |
| **C2** | **M** Gradient magnitude on logit $z_i$ is proportional to the post-softmax probability $x_i$, so confident tokens are nearly un-updatable | Eq. 25, App. C.3 | measured $\|\partial L/\partial z_i\|$ vs $x_i$ |
| **C3** | **M** DTO's refined policy has a far lower rejection rate than blind resampling (32.8% vs a theoretical 66.0%) | Tab. 3, Sec. 5.4 | measured rejection rate on our substrate |
| **C4** | **M** Inference-time gradient descent on Eq. 2 is dual to KL-regularised RL. The SDE $\frac{dx}{dt} = -\nabla\mathcal{L} + \sqrt{2}\epsilon$ has stationary distribution $\rho^\star = \arg\min \mathcal{L}_{PPO}$ | Thm. 4.1 | exact closed form on a finite space |
| **C5** | **H** ∇-Reasoner beats greedy / BoN / SC at **equal or fewer model calls** | Tab. 1, Fig. 3, Fig. 4 | accuracy at matched call budget |
| **C6** | **M** The three accelerations skip most of the work: caching 63.8% of grad calls, rollout reuse 74.1%, token selection 89.2% | App. D.2 | counters wired into our implementation |

C1–C4 are mechanism claims and are testable **exactly**, on small substrates, with no GPU noise. Those are where the real value is. C5 is the headline and is the one most at risk from resizing — the paper uses 7B/8B policies and a 4B reward model. We will be honest about the gap.

---

## Hardware and the resizing rule

2×T4, 15.6 GB each, compute capability 7.5. That means **fp16 and never bf16**.

The resizing rule from the paper-reproduction playbook: *do not run a shrunken copy of the original benchmark and report the noise.* Ask instead what the cheapest substrate is on which the claim is still falsifiable. For C1–C4 that substrate is a toy vocabulary where we can compute the answer in closed form and check it to machine precision. That is strictly **better** evidence than a 7B run, because there is no noise to hide in.

Deviations are tracked in a table at the end.

---
## Setup

Two cells. The first reports what hardware we landed on. The second installs and fixes seeds, and defines one helper used everywhere: `fig_show`, which saves every figure to `figs/` **and** renders it inline, so the notebook is readable top to bottom and the figures survive as files we can push to the repo.

In [2]:
import subprocess, sys, os, torch, platform
print("python :", sys.version.split()[0], "|", platform.platform()[:40])
print("torch  :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU{i}: {p.name}  {p.total_memory/1e9:.1f}GB  sm_{p.major}{p.minor}")
print("cwd    :", os.getcwd())
print("disk   :", subprocess.run("df -h . | tail -1", shell=True, capture_output=True, text=True).stdout.strip())
print("ram    :", subprocess.run("free -g | sed -n 2p", shell=True, capture_output=True, text=True).stdout.strip())
print("net    :", subprocess.run("timeout 8 curl -s -o /dev/null -w '%{http_code}' https://huggingface.co", shell=True, capture_output=True, text=True).stdout.strip())

python : 3.12.13 | Linux-6.12.90+-x86_64-with-glibc2.35
torch  : 2.10.0+cu128 | cuda: True
  GPU0: Tesla T4  15.6GB  sm_75
  GPU1: Tesla T4  15.6GB  sm_75
cwd    : /kaggle/working
disk   : /dev/loop1       20G  328K   20G   1% /kaggle/working
ram    : Mem:              31           1          25           0           4          29
net    : 200


In [2]:
%%capture
!pip -q install "transformers>=4.44,<5" accelerate datasets --upgrade

In [34]:
import os, math, json, time, random, warnings
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
torch.set_printoptions(precision=4, sci_mode=False, linewidth=120)
np.set_printoptions(precision=4, suppress=True, linewidth=120)

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
FIGS = ROOT / "figs"; FIGS.mkdir(exist_ok=True)
OUT  = ROOT / "out";  OUT.mkdir(exist_ok=True)

DEV   = "cuda" if torch.cuda.is_available() else "cpu"
# sm_75 (T4) has no bf16 tensor cores. fp16 everywhere, fp32/fp64 for exact work.
DTYPE = torch.float16

def seed_all(s=0):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

seed_all(0)

plt.rcParams.update({
    "figure.dpi": 96, "savefig.dpi": 130, "font.size": 9,
    "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
    "axes.spines.right": False, "figure.facecolor": "white",
    "axes.titlesize": 10, "axes.titleweight": "bold", "legend.frameon": False,
})

# Colour roles, kept identical across every figure in the notebook.
C = dict(base="#3B6FD4", dto="#D96A2B", exact="#1F9E6E",
         bad="#C4463F", grey="#8A8F98", accent="#7B5BD6")

INLINE_FIGS = True    # every figure is always written to figs/; this controls inline drawing

def fig_show(fig, name, inline=None):
    p = FIGS / f"{name}.png"
    fig.savefig(p, bbox_inches="tight")
    if (INLINE_FIGS if inline is None else inline): plt.show()
    else: plt.close(fig)
    print(f"  [fig] {p.name}")
    return p

def ok(cond, msg):
    """Assertion that prints its own verdict, so the notebook records what was checked."""
    print(f"  [{'PASS' if cond else 'FAIL'}] {msg}")
    assert cond, msg

def cached_model(name, make, train_fn):
    """
    Train once, cache the weights to out/. Survives kernel restarts, which
    matters here: an out-of-bounds index anywhere on the GPU kills the CUDA
    context for the whole process, and retraining the reward model is 10 minutes.
    Delete out/<name>.pt to force a retrain.
    """
    p = OUT / f"{name}.pt"
    m = make().to(DEV)
    if p.exists():
        m.load_state_dict(torch.load(p, map_location=DEV)); print(f"  [cache] loaded {name}")
    else:
        t0 = time.time(); train_fn(m); torch.save(m.state_dict(), p)
        print(f"  [cache] trained and saved {name} ({time.time()-t0:.1f}s)")
    m.eval()
    for q in m.parameters(): q.requires_grad_(False)
    return m

print(f"device {DEV} | dtype {DTYPE} | root {ROOT}")
print(f"figs -> {FIGS} | checkpoints -> {OUT}")

device cuda | dtype torch.float16 | root /kaggle/working
figs -> /kaggle/working/figs | checkpoints -> /kaggle/working/out


---
# Rung 0 — Which latent space?

Text is discrete. A length-$L$ sequence over a vocabulary $\mathcal{V}$ is one of $|\mathcal{V}|^L$ isolated points. You cannot take a derivative with respect to a point in a finite set, so you cannot do gradient descent on text.

The standard fix is to **relax**: replace each hard token with a point on the probability simplex, so the sequence becomes a point in a continuous space that has the real sequences sitting at its corners. Then you can differentiate.

$$y_i \in \Delta^{|\mathcal{V}|-1} \quad\text{parameterised by logits}\quad z_i \in \mathbb{R}^{|\mathcal{V}|},\qquad y_i = \mathrm{softmax}(z_i/\tau)$$

**This is the entire latent space of the paper.** $z \in \mathbb{R}^{|y| \times |\mathcal{V}|}$. Not hidden states, not the residual stream. The relaxed output text.

Two panels below make this concrete.

**Panel (a)** — one token, vocabulary of 3. The probability simplex is a triangle. The only three things you are allowed to actually *say* are the three corners. Every interior point is a relaxed token: a piece of text that is 60% "cat" and 40% "dog", which is not a sentence, but is a thing you can take a gradient of. DTO spends its whole life in that interior and only steps back to a corner when it is time to emit.

**Panel (b) and (c)** — two tokens, vocabulary of 2. Now the relaxed space is the unit square and the four real sequences `AA, AB, BA, BB` are its four corners. This is small enough to draw the **entire** objective landscape,

$$-\mathcal{L}(y) = \lambda\, r(y|x) + \log \pi_{LLM}(y|x)$$

which is Eq. 2 of the paper with the sign flipped so that higher is better. This is our reproduction of **Figure 1**.

The setup is rigged the way real decoding actually fails: the language model likes `AA` (so greedy decoding commits to `A` at step 1 and never recovers), while the reward is highest at `BB`. Greedy is myopic because it chooses token 1 knowing nothing about token 2.

- **Panel (b), zeroth-order.** Best-of-N can only ever land on corners. It throws $N$ darts at the four corners according to $\pi_{LLM}$ and keeps the best one. It never learns anything about the shape of the landscape between them.
- **Panel (c), first-order.** DTO reads $\nabla_z(\lambda r + \log \pi)$ and walks. It uses the *slope*, which is information a zeroth-order method structurally cannot access.

That difference is the whole paper.

In [5]:
# ---------------------------------------------------------------------------
# Rung 0: a toy world small enough to draw the ENTIRE objective landscape.
#   vocabulary {A, B}, sequence length 2  ->  relaxed space is the unit square
#   the 4 real sequences AA, AB, BA, BB are its 4 corners.
# ---------------------------------------------------------------------------
LAM  = 3.0        # lambda in Eq. 2: how hard we trade fluency for reward
TOK  = ["A", "B"]

# --- a two-token "language model" -------------------------------------------
# pi(y1)      = softmax(b1)
# pi(y2 | y1) = softmax(W @ y1)      <- W @ y1 is exactly an embedding lookup
#                                       when y1 is one-hot, and stays defined
#                                       (and differentiable) when it is not.
b1 = torch.tensor([1.0, 0.0])
W  = torch.tensor([[0.8, 0.0],
                   [0.0, 0.5]])

def log_pi(y1, y2):
    """log pi_LLM(y|x) for RELAXED y, via  sum_i y_i^T log Cat(pi(.|y_<i))  (Sec. 3.1)."""
    lp1 = F.log_softmax(b1, -1)
    lp2 = F.log_softmax(y1 @ W.T, -1)          # conditional depends on y1 continuously
    return (y1 * lp1).sum(-1) + (y2 * lp2).sum(-1)

# --- a differentiable reward model ------------------------------------------
# Deliberately multi-modal, like Fig. 1: one true optimum plus a decoy.
# Written on p = (P(y1=B), P(y2=B)), which are the square's coordinates.
BUMPS = [(1.00, 0.92, 0.88, 0.42),    # (amplitude, p1*, p2*, sigma)  <- true optimum, near BB
         (0.60, 0.10, 0.82, 0.20)]    #                               <- decoy, near AB
def reward(p1, p2):
    r = 0.0
    for a, c1, c2, s in BUMPS:
        r = r + a * torch.exp(-((p1 - c1) ** 2 + (p2 - c2) ** 2) / (2 * s ** 2))
    return r

def objective(y1, y2):
    """-L(y) = lambda*r + log pi. Higher is better. This is Eq. 2, sign-flipped."""
    return LAM * reward(y1[..., 1], y2[..., 1]) + log_pi(y1, y2)

# --- score all four real sequences (the corners) -----------------------------
def onehot(i): return torch.eye(2)[i]
corners = {}
for i, a in enumerate(TOK):
    for j, b in enumerate(TOK):
        y1, y2 = onehot(i), onehot(j)
        corners[a + b] = dict(
            obj=objective(y1, y2).item(),
            r=reward(y1[1], y2[1]).item(),
            logpi=log_pi(y1, y2).item(),
            p=(float(i), float(j)))

print("the four real sequences:")
print(f"  {'seq':<5}{'lambda*r':>10}{'log pi':>10}{'-L':>10}")
for k, v in corners.items():
    print(f"  {k:<5}{LAM*v['r']:>10.3f}{v['logpi']:>10.3f}{v['obj']:>10.3f}")

best_seq  = max(corners, key=lambda k: corners[k]["obj"])
greedy_1  = int(F.softmax(b1, -1).argmax())
greedy_2  = int(F.softmax(onehot(greedy_1) @ W.T, -1).argmax())
greedy_seq = TOK[greedy_1] + TOK[greedy_2]

print(f"\n  greedy decoding picks : {greedy_seq}   (-L = {corners[greedy_seq]['obj']:+.3f})")
print(f"  true optimum is       : {best_seq}   (-L = {corners[best_seq]['obj']:+.3f})")
ok(greedy_seq != best_seq, "toy world is rigged so greedy decoding is WRONG (myopia)")

the four real sequences:
  seq    lambda*r    log pi        -L
  AA        0.031    -0.684    -0.654
  AB        1.321    -1.484    -0.163
  BA        0.328    -2.287    -1.959
  BB        2.828    -1.787     1.041

  greedy decoding picks : AA   (-L = -0.654)
  true optimum is       : BB   (-L = +1.041)
  [PASS] toy world is rigged so greedy decoding is WRONG (myopia)


In [6]:
# ---------------------------------------------------------------------------
# Zeroth-order (Best-of-N) vs first-order (gradient ascent in logit space).
# ---------------------------------------------------------------------------
def sample_from_pi(g):
    """One full ancestral rollout from pi_LLM, i.e. what BoN / SC sample."""
    i = torch.multinomial(F.softmax(b1, -1), 1, generator=g).item()
    j = torch.multinomial(F.softmax(onehot(i) @ W.T, -1), 1, generator=g).item()
    return TOK[i] + TOK[j]

# --- Best-of-N: N darts at the corners, keep the highest reward --------------
N_BON, TRIALS = 8, 4000
g = torch.Generator().manual_seed(0)
hits, last_draw = 0, None
for t in range(TRIALS):
    draws = [sample_from_pi(g) for _ in range(N_BON)]
    pick  = max(draws, key=lambda s: corners[s]["r"])
    hits += (pick == best_seq)
    if t == 0: last_draw, first_pick = draws, pick
bon_rate = hits / TRIALS
print(f"Best-of-N (N={N_BON}), zeroth-order")
print(f"  one trial drew   : {last_draw}  -> kept {first_pick}")
print(f"  finds {best_seq} in {bon_rate*100:.1f}% of {TRIALS} trials")
print(f"  model calls      : {N_BON} full rollouts\n")

# --- DTO: gradient ascent on -L in logit space -------------------------------
# z is initialised from the LLM's own logits for the greedy rollout (Sec. 3.2).
z1 = b1.clone()
z2 = (onehot(greedy_1) @ W.T).clone()
z  = torch.stack([z1, z2]).requires_grad_(True)

opt, STEPS = torch.optim.Adam([z], lr=0.10), 60
path = []
for t in range(STEPS):
    y = F.softmax(z, -1)                       # relaxed tokens
    obj = objective(y[0], y[1])                # -L, we ascend it
    path.append((y[0, 1].item(), y[1, 1].item(), obj.item()))
    opt.zero_grad(); (-obj).backward(); opt.step()

y_fin = F.softmax(z.detach(), -1)
path.append((y_fin[0, 1].item(), y_fin[1, 1].item(), objective(y_fin[0], y_fin[1]).item()))
dto_seq = TOK[int(y_fin[0].argmax())] + TOK[int(y_fin[1].argmax())]

print(f"DTO (gradient ascent in logit space), first-order")
print(f"  start  p=({path[0][0]:.3f}, {path[0][1]:.3f})  -L={path[0][2]:+.3f}   decodes to {greedy_seq}")
print(f"  end    p=({path[-1][0]:.3f}, {path[-1][1]:.3f})  -L={path[-1][2]:+.3f}   decodes to {dto_seq}")
print(f"  model calls      : 1 rollout + {STEPS} gradient steps")
ok(dto_seq == best_seq, f"DTO escapes greedy's myopic {greedy_seq} and reaches the optimum {best_seq}")

Best-of-N (N=8), zeroth-order
  one trial drew   : ['BA', 'AA', 'AB', 'AA', 'AA', 'AA', 'AA', 'BB']  -> kept BB
  finds BB in 76.0% of 4000 trials
  model calls      : 8 full rollouts

DTO (gradient ascent in logit space), first-order
  start  p=(0.269, 0.310)  -L=-0.807   decodes to AA
  end    p=(0.883, 0.936)  -L=+1.210   decodes to BB
  model calls      : 1 rollout + 60 gradient steps
  [PASS] DTO escapes greedy's myopic AA and reaches the optimum BB


In [7]:
# ---------------------------------------------------------------------------
# Figure 1 reproduction.  (a) what the relaxed space IS,  (b) zeroth-order,  (c) first-order.
# ---------------------------------------------------------------------------
fig = plt.figure(figsize=(13.2, 4.1))

# ---- (a) one token, vocabulary of 3: the probability simplex ----------------
ax = fig.add_subplot(1, 3, 1)
V = np.array([[0, 0], [1, 0], [0.5, np.sqrt(3)/2]])          # triangle corners
ax.add_patch(plt.Polygon(V, closed=True, fc="#EEF2F9", ec=C["grey"], lw=1.2, zorder=0))
gen = torch.Generator().manual_seed(3)
pts = F.softmax(torch.randn(220, 3, generator=gen) * 1.5, -1).numpy()  # random relaxed tokens
xy  = pts @ V
ax.scatter(xy[:, 0], xy[:, 1], s=7, c=C["grey"], alpha=.45, lw=0, zorder=1)
for v, lab in zip(V, ['"cat"', '"dog"', '"the"']):
    ax.scatter(*v, s=120, c=C["base"], zorder=3, edgecolor="white", lw=1.4)
    ax.annotate(lab, v, textcoords="offset points",
                xytext=(0, 12 if v[1] > .1 else -17), ha="center", fontsize=9,
                color=C["base"], fontweight="bold")
mid = np.array([0.42, 0.34]) @ np.eye(2)
p_mid = np.array([0.42, 0.34, 0.24]) @ V
p_end = np.array([0.18, 0.62, 0.20]) @ V
ax.annotate("", xy=p_end, xytext=p_mid,
            arrowprops=dict(arrowstyle="-|>", color=C["dto"], lw=2.2, shrinkA=0, shrinkB=0), zorder=4)
ax.scatter(*p_mid, s=45, c=C["dto"], zorder=5, edgecolor="white", lw=1.2)
ax.text(0.50, 0.30, "a relaxed token\n(42% cat, 34% dog, 24% the)\nnot sayable, but differentiable",
        ha="center", va="top", fontsize=7.6, color="#444")
ax.set_title("(a) the latent space is the simplex\ncorners = real tokens, interior = relaxed")
ax.set_xlim(-.14, 1.14); ax.set_ylim(-.20, 1.02); ax.axis("off")

# ---- shared landscape for (b) and (c) --------------------------------------
gr = torch.linspace(0.001, 0.999, 260)
P1, P2 = torch.meshgrid(gr, gr, indexing="ij")
Y1 = torch.stack([1 - P1, P1], -1); Y2 = torch.stack([1 - P2, P2], -1)
Z = objective(Y1, Y2).detach().numpy()

def landscape(ax, title):
    ax.contourf(P1.numpy(), P2.numpy(), Z, levels=28, cmap="Oranges", alpha=.92, zorder=0)
    ax.contour(P1.numpy(), P2.numpy(), Z, levels=14, colors="white", linewidths=.5, alpha=.55, zorder=1)
    for name, v in corners.items():
        ax.scatter(*v["p"], s=130, marker="s", zorder=5,
                   c=(C["exact"] if name == best_seq else "#33373D"), edgecolor="white", lw=1.5)
        dx = -0.085 if v["p"][0] > .5 else 0.085
        dy = -0.075 if v["p"][1] > .5 else 0.075
        ax.annotate(name, (v["p"][0] + dx, v["p"][1] + dy), ha="center", va="center",
                    fontsize=9, fontweight="bold",
                    color=(C["exact"] if name == best_seq else "#33373D"), zorder=6)
    ax.set_xlabel(r"$P(y_1 = B)$"); ax.set_ylabel(r"$P(y_2 = B)$")
    ax.set_xlim(-.16, 1.16); ax.set_ylim(-.16, 1.16); ax.set_title(title); ax.grid(False)

# ---- (b) Best-of-N: darts at the corners only ------------------------------
ax = fig.add_subplot(1, 3, 2)
landscape(ax, "(b) zeroth-order (Best-of-N)\nonly ever lands on corners")
rng = np.random.default_rng(1)
for s in last_draw:
    px, py = corners[s]["p"]
    ax.scatter(px + rng.normal(0, .045), py + rng.normal(0, .045),
               s=34, c=C["base"], alpha=.85, zorder=4, edgecolor="white", lw=.8)
ax.scatter([], [], s=34, c=C["base"], label=f"{N_BON} samples ~ $\\pi_{{LLM}}$")
ax.plot([], [], ' ', label=f"finds {best_seq}: {bon_rate*100:.0f}% of trials")
ax.legend(loc="lower left", fontsize=7.4, labelspacing=.3)

# ---- (c) DTO: a walk through the interior -----------------------------------
ax = fig.add_subplot(1, 3, 3)
landscape(ax, "(c) first-order (DTO)\nfollows the gradient through the interior")
pa = np.array([(a, b) for a, b, _ in path])
ax.plot(pa[:, 0], pa[:, 1], color=C["dto"], lw=2.4, zorder=4)
ax.scatter(pa[0, 0], pa[0, 1], s=80, c="white", edgecolor=C["dto"], lw=2.2, zorder=6)
ax.scatter(pa[-1, 0], pa[-1, 1], s=80, marker="*", c=C["dto"], edgecolor="white", lw=1.2, zorder=6)
ax.annotate("init = greedy rollout's\nown logits", pa[0], textcoords="offset points",
            xytext=(14, -2), fontsize=7.4, color=C["dto"], va="center")
ax.plot([], [], color=C["dto"], lw=2.4, label=f"{STEPS} gradient steps")
ax.plot([], [], ' ', label=f"finds {best_seq}: deterministic")
ax.legend(loc="lower left", fontsize=7.4, labelspacing=.3)

fig.tight_layout()
fig_show(fig, "r0_latent_space")

  [fig] r0_latent_space.png


PosixPath('/kaggle/working/figs/r0_latent_space.png')

---
# Rung 1 — The straight-through estimator

Rung 0 cheated. It optimised a *relaxed* sequence and only rounded to real tokens at the very end. That is not what the paper does, and the difference matters.

The problem: the reward model and the language model are trained on **real text**. If you feed them a token that is 42% "cat" and 34% "dog", you are evaluating them far outside their training distribution, and their outputs become meaningless. So the forward pass has to see genuine one-hot tokens.

But `argmax` has zero gradient everywhere. It is piecewise constant, so backprop through it dies.

The **straight-through estimator** (Bengio et al. 2013) resolves this by using two different functions for the forward and backward pass. The paper writes it as:

$$y_i^{(t)} \;=\; \underbrace{\boldsymbol{\delta}_{\arg\max_j z^{(t)}_{ij}}}_{\text{hard one-hot}} \;+\; \mathrm{softmax}(z_i^{(t)}/\tau) \;-\; \underbrace{\mathrm{StopGrad}\big(\mathrm{softmax}(z_i^{(t)}/\tau)\big)}_{\text{cancels the term above, numerically}}$$

Read it as an algebraic trick:

- **Forward.** The last two terms are numerically identical, so they cancel exactly. What comes out is a pure one-hot vector. The models see real text.
- **Backward.** `StopGrad` contributes zero derivative and `δ` is a constant, so the only term that carries gradient is $\mathrm{softmax}(z_i/\tau)$. The Jacobian that backprop sees is the **softmax** Jacobian, which is smooth and non-zero.

So the forward pass is discrete and the backward pass is continuous. We are lying to the optimiser about what function it just computed, and the claim is that the lie is useful.

Three things get checked below, and all three are exact, not approximate:

1. the forward output is one-hot to machine precision,
2. the backward Jacobian equals $\frac{1}{\tau}(\mathrm{diag}(s) - s s^\top)$ analytically, and
3. plain `argmax` really does give an all-zero gradient, so the trick is load-bearing.

In [8]:
# ---------------------------------------------------------------------------
# The straight-through estimator, written exactly as the paper states it.
# ---------------------------------------------------------------------------
def st_onehot(z, tau=1.0):
    """
    y = delta_{argmax z}  +  softmax(z/tau)  -  StopGrad(softmax(z/tau))

    forward : hard one-hot (models see real text)
    backward: softmax Jacobian (gradient survives)
    """
    s    = F.softmax(z / tau, dim=-1)
    hard = F.one_hot(z.argmax(-1), z.shape[-1]).to(z.dtype)
    return hard + s - s.detach()

def softmax_jacobian(z, tau=1.0):
    """Analytic  d softmax(z/tau)_a / d z_b  =  (1/tau) * (diag(s) - s s^T)."""
    s = F.softmax(z / tau, -1)
    return (torch.diag(s) - torch.outer(s, s)) / tau

torch.manual_seed(0)
Vsz, TAU = 6, 1.0
z = torch.randn(Vsz, dtype=torch.float64, requires_grad=True)

# --- 1. forward is exactly one-hot ------------------------------------------
y = st_onehot(z, TAU)
print("z          :", z.detach().numpy())
print("softmax    :", F.softmax(z/TAU, -1).detach().numpy())
print("ST forward :", y.detach().numpy())
ok(torch.equal(y.detach(), F.one_hot(z.argmax(-1), Vsz).double()),
   "ST forward output is EXACTLY one-hot (the relaxation is invisible to the model)")

# --- 2. backward Jacobian is the softmax Jacobian ---------------------------
J_auto = torch.autograd.functional.jacobian(lambda t: st_onehot(t, TAU), z)
J_ana  = softmax_jacobian(z, TAU)
err    = (J_auto - J_ana).abs().max().item()
print(f"\nJacobian  d y / d z   (autograd vs analytic softmax Jacobian)")
print(f"  max abs difference : {err:.3e}")
ok(err < 1e-12, "ST backward Jacobian == (1/tau)(diag(s) - s s^T), to machine precision")

# --- 3. plain argmax is a dead end ------------------------------------------
# Worth being precise about HOW it fails. argmax does not return a small or zero
# gradient. It returns integer indices, so autograd never records an edge at all:
# the graph is severed and .backward() refuses to run.
z2   = z.detach().clone().requires_grad_(True)
hard = F.one_hot(z2.argmax(-1), Vsz).double()
print(f"\nplain argmax path:")
print(f"  z2.requires_grad          : {z2.requires_grad}")
print(f"  one_hot(argmax(z2)) .grad_fn : {hard.grad_fn}")
try:
    hard.sum().backward()
    outcome = f"gradient = {z2.grad.numpy()}"
except RuntimeError as e:
    outcome = f"RuntimeError: {e}"
print(f"  .backward()               -> {outcome}")
ok(hard.grad_fn is None and not hard.requires_grad,
   "plain argmax SEVERS the autograd graph entirely -> the ST trick is load-bearing")

# --- how temperature controls the sharpness of the backward signal ----------
print(f"\n{'tau':>6}{'||J||_F':>12}{'max s':>10}   (lower tau -> peakier softmax -> flatter Jacobian)")
for t in [2.0, 1.0, 0.5, 0.2, 0.1]:
    Jt = softmax_jacobian(z.detach(), t)
    print(f"{t:>6.1f}{Jt.norm().item():>12.4f}{F.softmax(z.detach()/t,-1).max().item():>10.4f}")

z          : [ 1.541  -0.2934 -2.1788  0.5684 -1.0845 -1.3986]
softmax    : [0.5926 0.0946 0.0144 0.2241 0.0429 0.0313]
ST forward : [1. 0. 0. 0. 0. 0.]
  [PASS] ST forward output is EXACTLY one-hot (the relaxation is invisible to the model)

Jacobian  d y / d z   (autograd vs analytic softmax Jacobian)
  max abs difference : 1.388e-17
  [PASS] ST backward Jacobian == (1/tau)(diag(s) - s s^T), to machine precision

plain argmax path:
  z2.requires_grad          : True
  one_hot(argmax(z2)) .grad_fn : None
  .backward()               -> RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn
  [PASS] plain argmax SEVERS the autograd graph entirely -> the ST trick is load-bearing

   tau     ||J||_F     max s   (lower tau -> peakier softmax -> flatter Jacobian)
   2.0      0.1955    0.3746
   1.0      0.3790    0.5926
   0.5      0.4482    0.8495
   0.2      0.0764    0.9922
   0.1      0.0012    0.9999


---
# Rung 2 — Eq. 25: why a confident token cannot be moved

The τ table above ended with something odd. As τ drops, the Frobenius norm of the backward Jacobian rises, peaks, and then **collapses to almost nothing**. At τ = 0.1 the softmax is 0.9999 confident and the gradient is $10^{-3}$, three orders of magnitude below its peak.

That is not a numerical artifact. It is App. C.3 of the paper, and it is the justification for one of the three acceleration tricks. The derivation is short. With $x = \mathrm{softmax}(z)$, the chain rule gives

$$\frac{\partial \mathcal{L}}{\partial z_i} \;=\; \frac{\partial \mathcal{L}}{\partial x}\frac{\partial x}{\partial z_i} \;=\; x_i\left(\left[\frac{\partial \mathcal{L}}{\partial x}\right]_i - x^\top \frac{\partial \mathcal{L}}{\partial x}\right) \tag{Eq. 25}$$

using the softmax Jacobian $\mathrm{diag}(x) - xx^\top$. Look at the structure. Every component of the logit gradient is **multiplied by its own post-softmax probability $x_i$**. So:

- For a token the model is unsure about, probability mass is spread out, many $x_i$ are moderate, and the gradient has real magnitude in many directions.
- For a token the model is confident about, one $x_i \approx 1$ and the rest are $\approx 0$. The near-zero entries are scaled to nothing by their own $x_i$. And the one surviving entry is scaled by $x_i \approx 1$ but multiplies $\big([\partial\mathcal{L}/\partial x]_i - x^\top \partial\mathcal{L}/\partial x\big)$, which also goes to zero, because when $x$ is nearly one-hot the weighted average $x^\top \partial\mathcal{L}/\partial x$ is nearly $[\partial\mathcal{L}/\partial x]_i$ itself.

**Both factors vanish at once.** The whole gradient vector dies. In the paper's words:

> *"When $x_i$ is small at the initialization, its underlying representation $z_i$ cannot be updated effectively."*

This is a real limitation, not a feature, but it has a useful consequence: if a confident token cannot be changed by gradient descent anyway, you should not spend a forward and backward pass through two large models trying. That is the **entropy criterion**, run DTO only when $H(z_1) > \epsilon_{ent}$, with $\epsilon_{ent} = 0.25$ in Tab. 4.

The paper states the threshold but never plots the curve it comes from. Below we do three things: verify Eq. 25 against autograd, measure the collapse over a real Qwen-sized vocabulary of 151,936, and then ask a question the paper leaves open — **how much gradient signal does the $\epsilon_{ent} = 0.25$ cut actually throw away?**

In [9]:
# ---------------------------------------------------------------------------
# Eq. 25 :  dL/dz_i = x_i * ( [dL/dx]_i  -  x^T dL/dx )
# ---------------------------------------------------------------------------
def eq25(x, dLdx):
    """Closed-form logit gradient from the post-softmax gradient."""
    return x * (dLdx - (x * dLdx).sum(-1, keepdim=True))

# --- verification against autograd ------------------------------------------
torch.manual_seed(1)
zc   = torch.randn(9, dtype=torch.float64, requires_grad=True)
gvec = torch.randn(9, dtype=torch.float64)                 # an arbitrary downstream dL/dx
xc   = F.softmax(zc, -1)
(xc * gvec).sum().backward()                               # so that dL/dx == gvec exactly
err  = (zc.grad - eq25(F.softmax(zc.detach(), -1), gvec)).abs().max().item()
print(f"Eq. 25 vs autograd : max abs difference = {err:.3e}")
ok(err < 1e-14, "Eq. 25 reproduces autograd exactly")

# --- the collapse, over a real Qwen-sized vocabulary -------------------------
VOCAB = 151_936
torch.manual_seed(0)
base = torch.randn(VOCAB, dtype=torch.float32)
g    = torch.randn(VOCAB, dtype=torch.float32)             # fixed downstream gradient

# Sweep an inverse-temperature c on the logits. Random Gaussian logits over a
# 152k vocabulary have a very small top-2 gap, so reaching the genuinely
# confident regime (H < 0.25 nats) takes c in the hundreds.
rows = []
for c in np.concatenate([np.linspace(0.05, 4.0, 30), np.geomspace(4.2, 400.0, 60)]):
    x  = F.softmax(base * float(c), -1)
    H  = float(-(x * (x + 1e-30).log()).sum())             # entropy in nats
    gz = eq25(x, g)
    rows.append((float(c), H, float(x.max()), float(gz.norm())))

Cs, Hs, pmax, gn = map(np.array, zip(*rows))
gn_rel = gn / gn.max()
H_peak = Hs[gn_rel.argmax()]

print(f"\n{'c':>8}{'entropy H':>11}{'max prob':>10}{'||dL/dz||':>12}{'rel':>8}")
for i in range(0, len(Hs), max(1, len(Hs)//14)):
    print(f"{Cs[i]:>8.2f}{Hs[i]:>11.4f}{pmax[i]:>10.5f}{gn[i]:>12.5f}{gn_rel[i]:>8.4f}")
print(f"{Cs[-1]:>8.2f}{Hs[-1]:>11.4f}{pmax[-1]:>10.5f}{gn[-1]:>12.5f}{gn_rel[-1]:>8.4f}")

# --- FINDING 1: Eq. 25 kills the gradient at BOTH ends, not just one ---------
# The paper discusses only the confident end. But x_i multiplies every
# component, so a near-UNIFORM token has every x_i ~ 1/|V| ~ 6.6e-6 and its
# gradient is crushed just as hard. The usable band is in the middle.
print(f"\nFINDING 1 -- the gradient is maximal in the MIDDLE of the entropy range.")
print(f"  peak at H = {H_peak:.2f} nats  ({pmax[gn_rel.argmax()]*100:.1f}% top-token prob)")
print(f"  near-uniform end  H = {Hs.max():6.2f} : {gn_rel[Hs.argmax()]*100:6.2f}% of peak")
print(f"  confident   end   H = {Hs.min():6.3f} : {gn_rel[Hs.argmin()]*100:6.2f}% of peak")
print(f"  The paper's epsilon_ent guards only the confident end. Eq. 25 says the")
print(f"  high-entropy end is just as dead, and nothing in Alg. 3 checks for it.")

# --- FINDING 2: what does epsilon_ent = 0.25 actually cost? ------------------
EPS_ENT = 0.25
lo      = gn_rel[Hs <= EPS_ENT]
at_thr  = gn_rel[np.abs(Hs - EPS_ENT).argmin()]
print(f"\nFINDING 2 -- the epsilon_ent = {EPS_ENT} cut (Tab. 4) is NOT free.")
print(f"  right at the threshold H = {EPS_ENT}, gradient norm is {at_thr*100:.1f}% of peak")
print(f"  across all skipped points: max {lo.max()*100:.2f}%, mean {lo.mean()*100:.2f}% of peak")
print(f"  So the threshold sits on the shoulder of the collapse, not past it.")
print(f"  It trades a real (~{at_thr*100:.0f}%) slice of signal for 89.2% fewer optimisation")
print(f"  steps (App. D.2). That is a good trade, but it is a trade.")

ok(gn_rel[Hs.argmin()] < 0.01,
   "gradient norm collapses to <1% of peak in the confident limit (App. C.3 confirmed)")
ok(gn_rel[Hs.argmax()] < 0.10,
   "gradient norm ALSO collapses to <10% of peak in the near-uniform limit (not in the paper)")

CONF_SWEEP = dict(c=Cs, H=Hs, pmax=pmax, gn=gn, gn_rel=gn_rel, H_peak=float(H_peak))

Eq. 25 vs autograd : max abs difference = 1.388e-17
  [PASS] Eq. 25 reproduces autograd exactly

       c  entropy H  max prob   ||dL/dz||     rel
    0.05    11.9300   0.00001     0.00257  0.0224
    0.87    11.5558   0.00024     0.00377  0.0328
    1.68    10.5115   0.00348     0.01004  0.0873
    2.50     8.7869   0.02617     0.03124  0.2716
    3.32     6.5563   0.10170     0.07175  0.6239
    4.20     4.3141   0.23706     0.10804  0.9395
    6.68     1.6155   0.53414     0.09092  0.7907
   10.61     0.8299   0.74284     0.04865  0.4231
   16.86     0.3959   0.89789     0.02459  0.2139
   26.80     0.1170   0.97726     0.00693  0.0603
   42.60     0.0158   0.99783     0.00073  0.0064
   67.71     0.0006   0.99994     0.00002  0.0002
  107.62     0.0000   1.00000     0.00000  0.0000
  171.05     0.0000   1.00000     0.00000  0.0000
  271.87     0.0000   1.00000     0.00000  0.0000
  400.00     0.0000   1.00000     0.00000  0.0000

FINDING 1 -- the gradient is maximal in the MIDDLE o

In [10]:
S = CONF_SWEEP
fig, axes = plt.subplots(1, 2, figsize=(11.2, 3.7))

ax = axes[0]
ax.plot(S["H"], S["gn_rel"], lw=2.2, color=C["base"])
ax.axvline(S["H_peak"], color=C["grey"], ls=":", lw=1.2)
ax.axvspan(-0.3, EPS_ENT, color=C["bad"], alpha=.13, lw=0)
ax.axvline(EPS_ENT, color=C["bad"], ls="--", lw=1.5)
ax.annotate(f"$\\epsilon_{{ent}}$ = {EPS_ENT}\nDTO skipped below here\n({at_thr*100:.0f}% of peak gradient)",
            (EPS_ENT, .62), xytext=(1.9, .70), fontsize=7.6, color=C["bad"],
            arrowprops=dict(arrowstyle="-|>", color=C["bad"], lw=1.2))
ax.annotate(f"peak at H={S['H_peak']:.1f}", (S["H_peak"], 1.0), xytext=(4.6, .95),
            fontsize=7.6, color=C["grey"])
ax.annotate("near-uniform:\nevery $x_i\\approx 1/|V|$,\ngradient dies too\n(not in the paper)",
            (S["H"].max(), S["gn_rel"][S["H"].argmax()]), xytext=(7.4, .40),
            fontsize=7.6, color=C["accent"],
            arrowprops=dict(arrowstyle="-|>", color=C["accent"], lw=1.2))
ax.set_xlabel("entropy of the token's logits $H(z)$   [nats]")
ax.set_ylabel(r"$\|\partial\mathcal{L}/\partial z\|_2$   (relative to peak)")
ax.set_title(f"Eq. 25: the usable gradient band\n|V| = {VOCAB:,}")
ax.set_xlim(-0.3, 12.3); ax.set_ylim(0, 1.06)

ax = axes[1]
ax.loglog(S["pmax"], np.maximum(S["gn_rel"], 1e-8), lw=2.2, color=C["dto"])
ax.set_xlabel("top-token probability  $\\max_i x_i$")
ax.set_ylabel(r"$\|\partial\mathcal{L}/\partial z\|_2$  (rel.)")
ax.set_title("the confident limit, on log axes\ngradient $\\to$ 0 as the model becomes sure")
ax.set_ylim(1e-8, 2)
for p_ref, lab in [(0.9, "90%"), (0.99, "99%"), (0.9999, "99.99%")]:
    i = np.abs(S["pmax"] - p_ref).argmin()
    ax.scatter(S["pmax"][i], S["gn_rel"][i], s=42, color=C["bad"], zorder=5, edgecolor="white", lw=1)
    ax.annotate(f"{lab} $\\to$ {S['gn_rel'][i]*100:.2g}%", (S["pmax"][i], S["gn_rel"][i]),
                textcoords="offset points", xytext=(-8, -14), fontsize=7.2,
                color=C["bad"], ha="right")

fig.tight_layout()
fig_show(fig, "r2_eq25_confidence")

  [fig] r2_eq25_confidence.png


PosixPath('/kaggle/working/figs/r2_eq25_confidence.png')

---
# Rung 3 — A substrate with exact ground truth

From here on we need a genuine autoregressive language model and a genuine reward model, because the remaining claims are about how gradient flows *through* them.

The temptation is to shrink GSM8K and run a 0.5B model on it. That is the wrong move. A resized benchmark gives you a noisy number on a hard task, and when the reproduction fails you cannot tell whether the method is wrong or whether you just did not have enough compute. The reproduction playbook says to ask a different question: **what is the cheapest substrate on which this claim is still falsifiable?**

Here it is **2-digit addition**, written as fixed-width strings:

```
prompt x = "47+58="        completion y = "105"
```

Vocabulary of 14. Every sequence has a correct answer known in closed form, so accuracy is exact with no grader, no parser, and no annotator noise. It has the property that matters: the answer is a short token sequence where **later digits depend on earlier ones through the carry**, which is precisely the long-range coupling that Claim C1 is about.

We train two models from scratch:

- **$\pi_{LLM}$**, a 4-layer causal transformer, deliberately under-trained so it gets a meaningful fraction of problems wrong. A perfect model leaves nothing for test-time scaling to do.
- **$r$**, an outcome reward model: the same architecture with a scalar head, reading the full `47+58=105` string and predicting whether the answer is correct. This is exactly the verifier of Cobbe et al. 2021, which is what the paper uses.

Both share the vocabulary, which the paper lists as a hard requirement of the method:

> *"The base and reward models are required to share the same vocabulary to allow for end-to-end logit optimization."*

The one thing that must be built carefully is the forward pass. It has to accept **relaxed** tokens, because that is the whole point. For hard tokens a transformer does an embedding lookup `E[i]`. For a relaxed token $y$ on the simplex, the same operation is the matrix product $y^\top E$, which agrees with the lookup exactly when $y$ is one-hot and stays differentiable when it is not. That single line is what makes gradients flow back into text.

In [36]:
# ---------------------------------------------------------------------------
# A small causal transformer, written out so nothing is hidden.
# The one non-standard requirement: the forward pass must accept RELAXED tokens.
# ---------------------------------------------------------------------------
ITOS = list("0123456789*=") + ["<pad>"]
STOI = {s: i for i, s in enumerate(ITOS)}
NV   = len(ITOS)                    # 13
PLEN, YLEN = 6, 4                   # "47*58=" then "2726"
SLEN = PLEN + YLEN                  # 10

def encode(s):  return torch.tensor([STOI[c] for c in s], dtype=torch.long)
def decode(t):  return "".join(ITOS[i] for i in t.tolist())

class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.dh = h, d // h
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, v, causal=True):
        B, T, D = v.shape
        q, k, vv = self.qkv(self.ln1(v)).chunk(3, -1)
        q, k, vv = (t.view(B, T, self.h, self.dh).transpose(1, 2) for t in (q, k, vv))
        a = F.scaled_dot_product_attention(q, k, vv, is_causal=causal)
        v = v + self.proj(a.transpose(1, 2).reshape(B, T, D))
        return v + self.mlp(self.ln2(v))

class TinyTransformer(nn.Module):
    """head='lm'  -> next-token logits.   head='reward' -> one scalar per sequence."""
    def __init__(self, d=128, h=4, layers=4, head="lm"):
        super().__init__()
        self.tok = nn.Embedding(NV, d)
        self.pos = nn.Parameter(torch.zeros(1, SLEN, d))
        self.blocks = nn.ModuleList([Block(d, h) for _ in range(layers)])
        self.lnf = nn.LayerNorm(d)
        self.head_kind = head
        self.out = nn.Linear(d, NV if head == "lm" else 1)

    def embed(self, ids=None, soft=None):
        """
        ids  : (B,T) int64          -> standard embedding lookup E[i]
        soft : (B,T,NV) simplex     -> y^T E, identical to the lookup when one-hot,
                                       and differentiable when it is not.
        """
        e = self.tok(ids) if soft is None else soft @ self.tok.weight
        return e + self.pos[:, :e.shape[1]]

    def forward(self, ids=None, soft=None):
        v = self.embed(ids, soft)
        causal = self.head_kind == "lm"
        for b in self.blocks:
            v = b(v, causal=causal)
        v = self.lnf(v)
        if self.head_kind == "lm":
            return self.out(v)                       # (B,T,NV)
        return self.out(v.mean(1)).squeeze(-1)       # (B,) scalar reward

# --- the relaxed-embedding identity, checked rather than asserted in prose ---
_m = TinyTransformer().eval()
_ids = torch.randint(0, NV, (3, SLEN))
with torch.no_grad():
    d_hard = _m.embed(ids=_ids)
    d_soft = _m.embed(soft=F.one_hot(_ids, NV).float())
ok(torch.allclose(d_hard, d_soft, atol=1e-6),
   "y^T E on a one-hot y is EXACTLY the embedding lookup -> relaxation is exact at the corners")
print(f"vocab {NV} | prompt {PLEN} + answer {YLEN} = {SLEN} tokens | "
      f"params {sum(p.numel() for p in _m.parameters())/1e3:.0f}k")

  [PASS] y^T E on a one-hot y is EXACTLY the embedding lookup -> relaxation is exact at the corners
vocab 13 | prompt 6 + answer 4 = 10 tokens | params 798k


In [37]:
# ---------------------------------------------------------------------------
# Build the dataset and train pi_LLM. 2-digit multiplication: hard enough for a
# 4-layer transformer to get wrong, and wrong in STRUCTURED, digit-level ways.
#
# We train TWO policies, and the difference between them decides Claim C1:
#   LM_SAT  plain cross-entropy       -> drives train loss to ~1e-4. Its softmax
#                                        SATURATES, so log pi has no usable
#                                        gradient left anywhere (see Rung 4).
#   LM_CAL  label smoothing eps=0.1   -> caps the max probability near 0.9, so
#                                        the likelihood term stays differentiable.
# Real LLMs see far more data than they can memorise and sit near LM_CAL.
# LM_CAL is the working policy for the rest of the notebook.
# ---------------------------------------------------------------------------
seed_all(0)
ALL = [(a, b) for a in range(100) for b in range(100)]
rng = np.random.default_rng(0); rng.shuffle(ALL)
N_TRAIN = 5000                                   # half held out, so it must generalise
TRAIN_P, TEST_P = ALL[:N_TRAIN], ALL[N_TRAIN:]

def as_str(a, b, ans=None):
    return f"{a:02d}*{b:02d}={(a*b) if ans is None else ans:04d}"
def batch_ids(pairs):
    return torch.stack([encode(as_str(a, b)) for a, b in pairs]).to(DEV)

X_TR, X_TE = batch_ids(TRAIN_P), batch_ids(TEST_P)
print("example rows:", [as_str(*p) for p in TRAIN_P[:4]])

# --- one guard used EVERYWHERE an answer becomes an integer ------------------
# The model can emit '*' or '=' in an answer slot. Those are token ids 10 and 11,
# so a naive base-10 decode overflows 9999 and any one_hot/gather on it indexes
# out of bounds -- which on CUDA is a device-side assert that kills the whole
# process. Every conversion goes through here.
DIGIT_W = torch.tensor([1000, 100, 10, 1], device=DEV)
def answer_index(seq):
    """(index in 0..9999, is_valid). Non-digit tokens make a row invalid."""
    d = seq[..., PLEN:]
    return (d.clamp(max=9) * DIGIT_W).sum(-1), (d < 10).all(-1)

def lm_loss(model, ids, ls=0.0):
    """Cross-entropy on the ANSWER positions only: the prompt is given, not predicted."""
    logits = model(ids=ids[:, :-1]); tgt = ids[:, 1:]
    return F.cross_entropy(logits[:, PLEN-1:].reshape(-1, NV),
                           tgt[:, PLEN-1:].reshape(-1), label_smoothing=ls)

@torch.no_grad()
def greedy_decode(model, ids):
    cur = ids[:, :PLEN].clone()
    for _ in range(YLEN):
        cur = torch.cat([cur, model(ids=cur)[:, -1].argmax(-1, keepdim=True)], 1)
    return cur

@torch.no_grad()
def exact_match(model, ids):
    return (greedy_decode(model, ids)[:, PLEN:] == ids[:, PLEN:]).all(1).float().mean().item()

@torch.no_grad()
def calibration(model, ids):
    """Mean top-token probability and mean entropy over the answer positions."""
    p = F.softmax(model(ids=ids[:, :-1])[:, PLEN-1:], -1)
    return p.max(-1).values.mean().item(), float(-(p * (p+1e-30).log()).sum(-1).mean())

def make_lm(): return TinyTransformer(d=128, h=4, layers=4, head="lm")
def lm_trainer(ls, steps=6000):
    def train(m):
        seed_all(0)
        opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=0.01)
        for step in range(steps):
            idx = torch.randint(0, len(X_TR), (256,), device=DEV)
            loss = lm_loss(m, X_TR[idx], ls=ls)
            opt.zero_grad(); loss.backward(); opt.step()
    return train

print("training two policies:")
LM_SAT = cached_model("lm_sat", make_lm, lm_trainer(0.0))
LM_CAL = cached_model("lm_cal", make_lm, lm_trainer(0.1))
for tag, m in [("LM_SAT", LM_SAT), ("LM_CAL", LM_CAL)]:
    pmx, ent = calibration(m, X_TE[:2048])
    print(f"  {tag:<8} raw train CE {lm_loss(m, X_TR[:2048]).item():8.5f} | "
          f"held-out EM {exact_match(m, X_TE)*100:5.2f}% | top-prob {pmx:.4f} | entropy {ent:.4f}")

LM         = LM_CAL                # the working policy from here on
GREEDY_ACC = exact_match(LM, X_TE)
print(f"\nworking policy = LM_CAL, greedy exact-match on {len(TEST_P)} held-out: {GREEDY_ACC*100:.2f}%")
ok(0.15 < GREEDY_ACC < 0.90, "pi_LLM is imperfect enough to leave real room for test-time scaling")
ok(calibration(LM_CAL, X_TE[:2048])[1] > 5 * calibration(LM_SAT, X_TE[:2048])[1],
   "LM_CAL is genuinely less saturated than LM_SAT -> log pi still carries gradient")

example rows: ['35*77=2695', '89*25=2225', '16*34=0544', '04*85=0340']
training two policies:
  [cache] loaded lm_sat
  [cache] loaded lm_cal
  LM_SAT   raw train CE  0.00008 | held-out EM 27.72% | top-prob 0.9461 | entropy 0.1385
  LM_CAL   raw train CE  0.10423 | held-out EM 28.90% | top-prob 0.7861 | entropy 0.8680

working policy = LM_CAL, greedy exact-match on 5000 held-out: 28.90%
  [PASS] pi_LLM is imperfect enough to leave real room for test-time scaling
  [PASS] LM_CAL is genuinely less saturated than LM_SAT -> log pi still carries gradient


In [38]:
# ---------------------------------------------------------------------------
# The outcome reward model r(y|x): reads "47*58=2726", predicts correct / not.
# This is the verifier of Cobbe et al. 2021, which is what the paper's RM is.
#
# NOTE (second version; v1 is in the report's eval-bugs section). v1 drew
# negatives ONLY from digit corruptions, near-misses and LM samples. It scored
# 93% balanced accuracy and was still USELESS as a ranker: its argmax over all
# 10,000 answers was right just 2% of the time, because 10,000-wide ranking
# queries it about candidates far outside the negative distribution it saw.
# v2 adds UNIFORM negatives over the whole answer space, which is the
# distribution the ranking metric actually asks about.
# ---------------------------------------------------------------------------
@torch.no_grad()
def sample_completions(model, prompts, n=1, temp=1.0, gen=None):
    cur = prompts.repeat_interleave(n, 0).clone()
    for _ in range(SLEN - prompts.shape[1]):
        p = F.softmax(model(ids=cur)[:, -1] / temp, -1)
        cur = torch.cat([cur, torch.multinomial(p, 1, generator=gen)], 1)
    return cur

def answers_of(ids):      return ids[:, PLEN:]
def is_correct(ids, ref): return (answers_of(ids) == answers_of(ref)).all(1)
def digits_of(v):         return torch.stack([(v // 10**k) % 10 for k in (3, 2, 1, 0)], 1)

seed_all(0)
X_ALL  = batch_ids(ALL)
AB_ALL = torch.tensor(ALL, device=DEV)                      # (10000, 2) the a,b of each row
g = torch.Generator(device=DEV).manual_seed(0)
POOL     = sample_completions(LM, X_ALL[:, :PLEN], n=4, temp=1.0, gen=g)
POOL_BAD = POOL[~is_correct(POOL, X_ALL.repeat_interleave(4, 0))]
print(f"LM sample pool: {len(POOL)} samples, {len(POOL_BAD)} wrong ({len(POOL_BAD)/len(POOL)*100:.1f}%)")

# uniform is first because it is what makes the RM a usable RANKER, not just a
# usable classifier.
NEG_MIX = dict(uniform=0.34, corrupt=0.24, nearmiss=0.22, lm=0.20)
CUTS = torch.tensor(np.cumsum(list(NEG_MIX.values()))[:-1], device=DEV).float()

def make_rm_batch(bs, gen=None):
    idx  = torch.randint(0, len(X_ALL), (bs,), device=DEV, generator=gen)
    pos  = X_ALL[idx]; half = bs // 2; nb = bs - half
    neg  = pos[half:].clone()
    true = AB_ALL[idx[half:], 0] * AB_ALL[idx[half:], 1]
    src  = torch.bucketize(torch.rand(nb, device=DEV, generator=gen), CUTS)

    m = src == 0                                    # uniform over the whole answer space
    if m.any():
        neg[m, PLEN:] = digits_of(torch.randint(0, 10000, (int(m.sum()),), device=DEV, generator=gen))
    m = src == 1                                    # corrupt 1-2 digits of the truth
    if m.any():
        sub = neg[m]
        for _ in range(2):
            pi  = torch.randint(PLEN, SLEN, (len(sub),), device=DEV, generator=gen)
            val = torch.randint(0, 10, (len(sub),), device=DEV, generator=gen)
            hit = torch.rand(len(sub), device=DEV, generator=gen) < 0.65
            sub[torch.arange(len(sub), device=DEV)[hit], pi[hit]] = val[hit]
        neg[m] = sub
    m = src == 2                                    # near-miss: true product +/- small delta
    if m.any():
        d = torch.randint(1, 60, (int(m.sum()),), device=DEV, generator=gen)
        s = torch.where(torch.rand(int(m.sum()), device=DEV, generator=gen) < .5, -1, 1)
        neg[m, PLEN:] = digits_of((true[m] + s*d).clamp(0, 9999))
    m = src == 3                                    # an actual wrong LM sample
    if m.any():
        pick = torch.randint(0, len(POOL_BAD), (int(m.sum()),), device=DEV, generator=gen)
        neg[m, PLEN:] = POOL_BAD[pick][:, PLEN:]

    x = torch.cat([pos[:half], neg], 0)
    return x, is_correct(x, torch.cat([pos[:half], pos[half:]], 0)).float()

RM_STEPS = 14000
def make_rm(): return TinyTransformer(d=192, h=6, layers=6, head="reward")
def rm_train(m):
    seed_all(1)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=0.01)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, RM_STEPS, eta_min=3e-5)
    g   = torch.Generator(device=DEV).manual_seed(1)
    for step in range(RM_STEPS):
        x, lab = make_rm_batch(384, g)
        loss = F.binary_cross_entropy_with_logits(m(ids=x), lab)
        opt.zero_grad(); loss.backward(); opt.step(); sch.step()
        if (step+1) % 3500 == 0: print(f"    rm step {step+1:>6}  loss {loss.item():.4f}")

RM = cached_model("rm_v2", make_rm, rm_train)
with torch.no_grad():
    xv, lv = make_rm_batch(8192, torch.Generator(device=DEV).manual_seed(999))
    bal = (((RM(ids=xv) > 0).float()) == lv).float().mean().item()
print(f"reward model | params {sum(p.numel() for p in RM.parameters())/1e6:.2f}M | "
      f"balanced acc {bal*100:.1f}%")
print("the RANKING test that actually matters is the next cell.")

LM sample pool: 40000 samples, 23492 wrong (58.7%)
  [cache] loaded rm_v2
reward model | params 2.67M | balanced acc 98.3%
the RANKING test that actually matters is the next cell.


---
## Bracketing the instrument before spending any compute

The reproduction playbook has one rule that saves more projects than any other: **build the measurement and gate on it, before running the sweep.** If a metric runs through a learned component, measure its ceiling and its floor first, because the method can only ever score inside that bracket.

DTO's ceiling is set by the reward model. DTO does not maximise correctness. It maximises

$$-\mathcal{L}(y) = \lambda\, r(y|x) + \log \pi_{LLM}(y|x)$$

and $r$ is a 93%-accurate neural network, not an oracle. So if the reward model's own argmax is wrong, a *perfect* optimiser would confidently return a wrong answer. Any accuracy DTO fails to reach might be the optimiser's fault or the objective's fault, and those are completely different conclusions.

Our substrate lets us separate them exactly. There are only $10^4$ possible answers, so for each problem we can score **every single one** under both $r$ and $\log\pi_{LLM}$, and then compute the exact global maximiser of Eq. 2 for any $\lambda$ by brute force. The paper cannot do this: their space is $151936^{1024}$.

That gives us three reference lines that the rest of the notebook is read against:

| line | meaning |
|---|---|
| $\lambda \to 0$ | pure likelihood. The MAP sequence under $\pi_{LLM}$, which is what decoding is trying to find. |
| $\lambda \to \infty$ | pure reward. The reward model's own argmax, ignoring fluency. **This is the hard ceiling of any reward-guided method.** |
| the curve between | the exact optimum of Eq. 2 at each $\lambda$: the target DTO's gradient descent is chasing |

Everything DTO achieves later gets reported as a fraction of this curve, so an **optimiser gap** can never be confused with an **objective gap**. It also lets us watch reward hacking happen directly: as $\lambda$ rises, the objective stops caring about $\pi_{LLM}$, and we can see exactly where that starts to cost accuracy.

In [39]:
# ---------------------------------------------------------------------------
# Brute-force the EXACT global optimum of Eq. 2 over the whole answer space.
# For N_PROB held-out problems x 10,000 candidate answers, score every one under
# both r and log pi. Then argmax(lambda*r + log pi) is exact for any lambda.
# The paper cannot do this: their space is 151936^1024.
# ---------------------------------------------------------------------------
N_PROB = 200
seed_all(0)
probe_idx = torch.randperm(len(TEST_P))[:N_PROB]
PROBE     = X_TE[probe_idx.to(DEV)]                       # (N_PROB, SLEN) ground truth
print(f"probing {N_PROB} held-out problems x 10,000 candidate answers "
      f"= {N_PROB*10000:,} scored sequences")

cand = torch.arange(10000, device=DEV)
CAND = torch.stack([(cand // 10**k) % 10 for k in (3, 2, 1, 0)], 1)   # (10000, YLEN)

@torch.no_grad()
def score_all(prompt_ids):
    """For ONE prompt, return (r, log_pi) over all 10,000 candidate answers."""
    seqs = torch.cat([prompt_ids.view(1, -1).expand(10000, -1), CAND], 1)
    rs, lps = [], []
    for s in range(0, 10000, 2500):
        ch = seqs[s:s+2500]
        rs.append(torch.sigmoid(RM(ids=ch)))
        lg = F.log_softmax(LM(ids=ch[:, :-1])[:, PLEN-1:], -1)
        lps.append(lg.gather(-1, ch[:, PLEN:].unsqueeze(-1)).squeeze(-1).sum(-1))
    return torch.cat(rs), torch.cat(lps)

t0 = time.time()
R_ALL  = torch.zeros(N_PROB, 10000, device=DEV)
LP_ALL = torch.zeros(N_PROB, 10000, device=DEV)
for i in range(N_PROB):
    R_ALL[i], LP_ALL[i] = score_all(PROBE[i, :PLEN])
TRUE_IDX = (PROBE[:, PLEN:] * torch.tensor([1000, 100, 10, 1], device=DEV)).sum(1)
print(f"scored in {time.time()-t0:.1f}s")

def exact_opt_acc(lam):
    """Accuracy of the EXACT global maximiser of  lambda*r + log pi."""
    return ((lam * R_ALL + LP_ALL).argmax(1) == TRUE_IDX).float().mean().item()

LAMBDAS = np.concatenate([[0.0], np.geomspace(0.05, 600, 40)])
CEIL    = np.array([exact_opt_acc(float(l)) for l in LAMBDAS])

MAP_ACC   = exact_opt_acc(0.0)          # lambda -> 0 : pure likelihood (the MAP sequence)
RONLY_ACC = exact_opt_acc(1e9)          # lambda -> inf: pure reward (the RM's own argmax)
BEST_L    = float(LAMBDAS[CEIL.argmax()]); BEST_ACC = float(CEIL.max())
LAM_MAIN  = BEST_L                      # every later experiment uses this lambda

print(f"\n{'lambda':>9}{'exact-opt acc':>16}")
for l, a in zip(LAMBDAS[::4], CEIL[::4]): print(f"{l:>9.3g}{a*100:>15.1f}%")
print(f"{'inf':>9}{RONLY_ACC*100:>15.1f}%")

print(f"\n--- the bracket ---------------------------------------------------")
print(f"  FLOOR   random 4-digit answer            :   0.01%")
print(f"  greedy decoding (argmax step by step)    : {GREEDY_ACC*100:6.2f}%   <- what we must beat")
print(f"  lambda->0   MAP sequence under pi_LLM    : {MAP_ACC*100:6.2f}%   <- best possible WITHOUT reward")
print(f"  lambda={BEST_L:<5.3g} exact optimum of Eq. 2      : {BEST_ACC*100:6.2f}%   <- CEILING for DTO")
print(f"  lambda->inf reward model's own argmax    : {RONLY_ACC*100:6.2f}%   <- reward hacking limit")
print(f"  headroom over greedy                     : {(BEST_ACC-GREEDY_ACC)*100:+6.2f} points")
print(f"-------------------------------------------------------------------")
print(f"  every later experiment uses lambda = LAM_MAIN = {LAM_MAIN:.3g}")
ok(BEST_ACC > GREEDY_ACC + 0.02,
   "the Eq. 2 objective genuinely has headroom over greedy -> the experiment is worth running")
ok(BEST_ACC > RONLY_ACC,
   "the log-pi regulariser BEATS pure reward maximisation -> Eq. 2's two terms both matter")

probing 200 held-out problems x 10,000 candidate answers = 2,000,000 scored sequences
scored in 83.8s

   lambda   exact-opt acc
        0           27.5%
    0.103           27.5%
     0.27           27.5%
    0.707           28.5%
     1.85           32.0%
     4.86           35.5%
     12.7           36.0%
     33.3           37.5%
     87.4           39.5%
      229           35.0%
      600           27.0%
      inf           17.0%

--- the bracket ---------------------------------------------------
  FLOOR   random 4-digit answer            :   0.01%
  greedy decoding (argmax step by step)    :  28.90%   <- what we must beat
  lambda->0   MAP sequence under pi_LLM    :  27.50%   <- best possible WITHOUT reward
  lambda=87.4  exact optimum of Eq. 2      :  39.50%   <- CEILING for DTO
  lambda->inf reward model's own argmax    :  17.00%   <- reward hacking limit
  headroom over greedy                     : +10.60 points
--------------------------------------------------------------

---
# Rung 4 — Proposition C.1, and the claim the paper never measures

This is the heart of the method, and the place where ∇-Reasoner differs from a decade of gradient-based controlled decoding.

Autoregressive generation is strictly left to right. Token 3 is chosen knowing tokens 1 and 2, and nothing can ever flow backwards. That is the structural reason greedy decoding is myopic, and it is what Bachmann & Nagarajan call the pitfall of next-token prediction: an early token commits you to a path and no later evidence can undo it.

Proposition C.1 says that **the gradient does not have this restriction.** Differentiating Eq. 2 with respect to the $l$-th token splits into exactly three pieces:

$$\frac{\partial \mathcal{L}}{\partial y_l} \;=\; \underbrace{\boldsymbol{\delta}_{\text{prefix}}}_{\text{what came before}} \;+\; \underbrace{\boldsymbol{\delta}_{\text{postfix}}}_{\text{what comes after}} \;+\; \lambda\,\underbrace{\boldsymbol{\delta}_{\text{reward}}}_{\text{the whole sequence's score}}$$

with

$$\boldsymbol{\delta}_{\text{prefix}} = -\log \mathrm{Cat}\big(\pi_{LLM}(\cdot|y_{\leq l-1}, x)\big) \tag{Eq. 4}$$
$$\boldsymbol{\delta}_{\text{postfix}} = -\sum_{i=l+1}^{|y|} \frac{\partial \log \mathrm{Cat}(\pi_{LLM}(\cdot|y_{\leq i-1},x))}{\partial y_l}\, y_i \tag{Eq. 5}$$
$$\boldsymbol{\delta}_{\text{reward}} = -\sum_{i=l}^{|y|} \frac{\partial r(y_{\leq i}|x)}{\partial y_l} \tag{Eq. 6}$$

Read the middle one carefully, because it is the whole argument. $\boldsymbol{\delta}_{\text{postfix}}$ sums over tokens **after** $l$. It is the answer to "if I changed token 3, how much worse would the model's own predictions of tokens 4, 5, 6 become?" That signal travels backwards along the sequence, through the attention that later positions pay to earlier ones. It is information that autoregressive decoding structurally cannot use.

And this is exactly where the paper plants its flag against prior work (Remark C.2):

> *"Our proposed DTO fundamentally differs from previous works that utilize gradients for controlled generation, where $\boldsymbol{\delta}_{\text{postfix}}$ is often detached from the computational graph, and only prior context is used to guide subsequent token prediction."*

That is a strong, specific, falsifiable claim about a mechanism. **It also has no experiment attached to it anywhere in the paper.** There is no ablation table, no figure, no number. We are going to supply one:

1. verify the three-way split against autograd to machine precision,
2. verify $\boldsymbol{\delta}_{\text{prefix}}$ equals the closed form of Eq. 4, not merely "some gradient",
3. measure how big $\boldsymbol{\delta}_{\text{postfix}}$ actually is relative to the other two terms, position by position, and
4. run DTO twice, once complete and once with $\boldsymbol{\delta}_{\text{postfix}}$ detached, which turns the method into the prior work it claims to beat, and measure what the difference is worth.

Step 4 is the experiment the paper is missing.

In [15]:
# ---------------------------------------------------------------------------
# Proposition C.1, verified in float64 on CPU so the check is exact, not "close".
# Run on BOTH policies, because saturation turns out to decide whether the
# bidirectional term exists at all.
# ---------------------------------------------------------------------------
def to64(m, **kw):
    c = TinyTransformer(**kw).double().cpu()
    c.load_state_dict({k: v.double().cpu() for k, v in m.state_dict().items()})
    c.eval()
    for p in c.parameters(): p.requires_grad_(False)
    return c

LMc = {"LM_CAL": to64(LM_CAL, d=128, h=4, layers=4, head="lm"),
       "LM_SAT": to64(LM_SAT, d=128, h=4, layers=4, head="lm")}
RM64 = to64(RM, d=192, h=6, layers=6, head="reward")

def full_soft(prompt_ids, y_soft):
    oh = F.one_hot(prompt_ids, NV).to(y_soft.dtype)
    return torch.cat([oh, y_soft], 0).unsqueeze(0)

def logpi_terms(lm, prompt_ids, y_soft):
    """Per-position  log pi(y_i | y_<i, x)  as a length-YLEN vector."""
    logits = lm(soft=full_soft(prompt_ids, y_soft))[0]
    lp     = F.log_softmax(logits[PLEN-1:SLEN-1], -1)
    return (y_soft * lp).sum(-1)

def reward_of(rm, prompt_ids, y_soft):
    return torch.sigmoid(rm(soft=full_soft(prompt_ids, y_soft))[0])

LAM_CHK = 5.0
seed_all(0)
pid = X_TE[3, :PLEN].cpu()
DEC = {}

for tag, lm64 in LMc.items():
    with torch.no_grad():
        y0 = greedy_decode(LM_CAL if tag == "LM_CAL" else LM_SAT, X_TE[3:4])[0, PLEN:].cpu()
        z0 = lm64(soft=full_soft(pid, F.one_hot(y0, NV).double()))[0][PLEN-1:SLEN-1]

    def grads_for(loss_fn):
        y = F.one_hot(y0, NV).double().requires_grad_(True)
        loss_fn(y).backward()
        return y.grad.clone()

    g_total = grads_for(lambda y: -LAM_CHK*reward_of(RM64, pid, y) - logpi_terms(lm64, pid, y).sum())
    d_pre   = torch.stack([grads_for(lambda y, l=l: -logpi_terms(lm64, pid, y)[l])[l] for l in range(YLEN)])
    d_post  = torch.stack([grads_for(lambda y, l=l: -logpi_terms(lm64, pid, y)[l+1:].sum())[l]
                           if l+1 < YLEN else torch.zeros(NV, dtype=torch.float64) for l in range(YLEN)])
    d_rew   = torch.stack([grads_for(lambda y: -reward_of(RM64, pid, y))[l] for l in range(YLEN)])

    err = (g_total - (d_pre + d_post + LAM_CHK*d_rew)).abs().max().item()
    with torch.no_grad():
        cf = -F.log_softmax(z0, -1)
    err4 = (d_pre - cf).abs().max().item()
    print(f"[{tag}]  problem {decode(pid)}  greedy answer {decode(y0)}  (truth {decode(X_TE[3, PLEN:].cpu())})")
    print(f"   Prop. C.1 reconstruction error : {err:.3e}")
    print(f"   Eq. 4 closed form vs autograd  : {err4:.3e}")
    ok(err  < 1e-9, f"[{tag}] Prop. C.1 three-way decomposition is EXACT")
    ok(err4 < 1e-9, f"[{tag}] delta_prefix IS exactly -log Cat(pi_LLM(.|y_<=l-1,x)), not merely 'a gradient'")

    # ---- the correction that matters ---------------------------------------
    # ||delta|| on y OVERSTATES delta_prefix. delta_prefix = -log Cat(.) is a
    # vector of negative log-probs, so it is huge (~20 per entry) but nearly
    # CONSTANT across the vocabulary. Eq. 25 then projects out exactly the
    # component along x: g_z = x * (delta - x . delta). A constant vector has
    # zero projection. So we must compare the terms AFTER that projection,
    # which is what actually reaches the optimiser.
    x0 = F.softmax(z0, -1)
    proj = lambda d: (x0 * (d - (x0*d).sum(-1, keepdim=True))).norm(dim=-1)
    DEC[tag] = dict(
        raw =dict(pre=d_pre.norm(dim=-1).numpy(), post=d_post.norm(dim=-1).numpy(),
                  rew=(LAM_CHK*d_rew).norm(dim=-1).numpy()),
        prj =dict(pre=proj(d_pre).numpy(), post=proj(d_post).numpy(), rew=proj(LAM_CHK*d_rew).numpy()),
        ent =float(-(x0*(x0+1e-30).log()).sum(-1).mean()))

    for space, lab in [("raw", "on y (raw)"), ("prj", "on z (after Eq. 25 projection)")]:
        D = DEC[tag][space]
        print(f"   {lab}")
        print(f"   {'pos':>5}{'|d_prefix|':>13}{'|d_postfix|':>14}{'lam|d_reward|':>16}{'postfix share':>15}")
        for l in range(YLEN):
            t = D['pre'][l] + D['post'][l] + D['rew'][l]
            print(f"   {l:>5}{D['pre'][l]:>13.5f}{D['post'][l]:>14.5f}{D['rew'][l]:>16.5f}"
                  f"{D['post'][l]/max(t,1e-30)*100:>14.2f}%")
    print(f"   mean logit entropy of this policy: {DEC[tag]['ent']:.4f} nats\n")

ok(DEC["LM_CAL"]["prj"]["post"][:YLEN-1].min() > 1e-8,
   "[LM_CAL] delta_postfix is NON-ZERO for every non-final token -> gradient flows BACKWARDS (C1)")
ok(DEC["LM_CAL"]["raw"]["post"][YLEN-1] == 0.0,
   "delta_postfix is exactly zero for the LAST token, which has no future (sanity check)")
print(f">>> delta_postfix (projected, summed over positions)")
print(f"      LM_CAL (entropy {DEC['LM_CAL']['ent']:.3f}) : {DEC['LM_CAL']['prj']['post'].sum():.6f}")
print(f"      LM_SAT (entropy {DEC['LM_SAT']['ent']:.3f}) : {DEC['LM_SAT']['prj']['post'].sum():.6f}")
print(f"      ratio  : {DEC['LM_CAL']['prj']['post'].sum()/max(DEC['LM_SAT']['prj']['post'].sum(),1e-30):.1f}x")

[LM_CAL]  problem 42*20=  greedy answer 0840  (truth 0840)
   Prop. C.1 reconstruction error : 1.332e-14
   Eq. 4 closed form vs autograd  : 0.000e+00
  [PASS] [LM_CAL] Prop. C.1 three-way decomposition is EXACT
  [PASS] [LM_CAL] delta_prefix IS exactly -log Cat(pi_LLM(.|y_<=l-1,x)), not merely 'a gradient'
   on y (raw)
     pos   |d_prefix|   |d_postfix|   lam|d_reward|  postfix share
       0     16.74096       1.17856        17.47805          3.33%
       1     16.88513       0.80344        19.30316          2.17%
       2     16.87071       0.04619         6.88974          0.19%
       3     16.93504       0.00000         2.86800          0.00%
   on z (after Eq. 25 projection)
     pos   |d_prefix|   |d_postfix|   lam|d_reward|  postfix share
       0      0.42989       0.00848         0.18432          1.36%
       1      0.46952       0.03222         0.91476          2.27%
       2      0.44493       0.00202         0.22215          0.30%
       3      0.41260       0.00000     

---
# Rung 5 — Algorithm 2: Differentiable Textual Optimization

Everything so far assembles into the paper's inner loop. Algorithm 2 verbatim:

```
Require: prefix x, initial logits z, model pi_LLM, reward r, steps T
 1:  z(1) <- z
 2:  for t = 1 ... T do
 3:      for every i = 1 ... |y| do
 4:          j*    <- argmax_j z_ij(t)
 5:          y_i(t) <- delta_{j*} + softmax(z_i(t)/tau) - StopGrad(softmax(z_i(t)/tau))
 6:      end for
 7:      L_nll    = - sum_i log pi_LLM( y(t) | y(t)_{<=i-1}, x )
 8:      L_reward = - r( y(t) | x )
 9:      L        = L_nll + lambda * L_reward
10:      z(t+1)   <- z(t) - eta * grad_z L
11:  end for
12:  return z(T)
```

Lines 3 to 6 are Rung 1. Lines 7 to 9 are Eq. 2. Line 10 is ordinary gradient descent, with AdamW, a cosine schedule, $\eta = 0.01$ decaying to $0.001$, and $T = 20$ (Tab. 4).

The initial $z$ matters and is easy to miss. It is **not** random and **not** zero. It is the language model's own pre-softmax logits from the rollout it just produced. So DTO starts exactly where normal decoding would have stopped, and every gradient step is a departure from the base policy's answer. That is why the method can be dropped into decoding without retraining anything.

### How to ablate $\boldsymbol{\delta}_{\text{postfix}}$

The ablation from Rung 4 has a clean implementation, which is worth seeing because it shows precisely what "detached from the computational graph" means in Remark C.2.

The log-likelihood term at position $i$ is $y_i^\top \log\mathrm{Cat}(\pi_{LLM}(\cdot|y_{<i},x))$. Gradient reaches $y_l$ by two separate routes:

- through the $y_i$ factor, which only touches $l = i$. This is $\boldsymbol{\delta}_{\text{prefix}}$.
- through the $\log\mathrm{Cat}(\cdot)$ factor, which depends on the whole context $y_{<i}$, so it touches every $l < i$. This is $\boldsymbol{\delta}_{\text{postfix}}$.

So to remove $\boldsymbol{\delta}_{\text{postfix}}$ and nothing else, you feed the model a **detached** copy of $y$ when computing the conditional distribution, while keeping the $y_i$ factor live:

```python
logits = LM(soft = cat([prompt, y.detach()]))   # context contributes no gradient
lp     = log_softmax(logits).detach()
L_nll  = -(y * lp).sum()                        # gradient only via the y_i factor
```

One `.detach()` converts ∇-Reasoner into the prior work it distinguishes itself from. That makes it a clean single-variable ablation, which is exactly what the playbook says to look for.

In [40]:
# ---------------------------------------------------------------------------
# Algorithm 2: Differentiable Textual Optimization, batched over problems.
# Written for an ARBITRARY prefix length, because Algorithm 1 calls it again at
# every decoding step with one more token fixed.
# ---------------------------------------------------------------------------
CALLS = dict(fwd=0, bwd=0)                 # model-call accounting, used for Fig. 3 / Tab. 5
def reset_calls(): CALLS.update(fwd=0, bwd=0)

@torch.no_grad()
def rollout_with_logits(prefix, temp=None, gen=None):
    """
    Autoregressive rollout to full length, returning BOTH the tokens and the
    per-token pre-softmax logits. Those logits are DTO's z^(0) (Sec. 3.2).
    temp=None -> greedy.
    An ALREADY-FULL prefix is legal and yields zero new tokens: Alg. 3 hits this
    at the last decoding position, where there is no continuation left to sample.
    """
    cur, zs = prefix.clone(), []
    for _ in range(SLEN - prefix.shape[1]):
        lg = LM(ids=cur)[:, -1]; CALLS["fwd"] += 1
        zs.append(lg)
        nxt = (lg.argmax(-1, keepdim=True) if temp is None
               else torch.multinomial(F.softmax(lg / temp, -1), 1, generator=gen))
        cur = torch.cat([cur, nxt], 1)
    z = (torch.stack(zs, 1) if zs
         else torch.empty(len(prefix), 0, NV, device=prefix.device))
    return cur, z                                     # (B,SLEN), (B,rem,NV)

def dto_loss(prefix, y, lam, use_postfix=True, use_reward=True):
    """Eq. 2 on a batch of relaxed suffixes. Returns (L, L_nll, r)."""
    P    = prefix.shape[1]
    oh   = F.one_hot(prefix, NV).to(y.dtype)
    full = torch.cat([oh, y], 1)
    if use_postfix:
        logits = LM(soft=full)                        # context is differentiable
    else:
        # Remark C.2: prior work detaches delta_postfix. One line does it.
        logits = LM(soft=torch.cat([oh, y.detach()], 1)).detach()
    CALLS["fwd"] += 1
    lp    = F.log_softmax(logits[:, P-1:SLEN-1], -1)
    L_nll = -(y * lp).sum(-1).sum(-1)                 # (B,)
    r     = torch.sigmoid(RM(soft=full)); CALLS["fwd"] += 1
    if not use_reward: r = r.detach()
    return L_nll - lam * r, L_nll, r

def DTO(prefix, z0, lam=5.0, T=20, tau=1.0, lr=0.01, min_lr=1e-3,
        use_postfix=True, use_reward=True, trace=False):
    """
    Algorithm 2. Returns refined logits z^(T).
    Hyperparameters follow Tab. 4: AdamW, cosine schedule, lr 0.01 -> 0.001, T=20.
    """
    z   = z0.clone().float().requires_grad_(True)
    opt = torch.optim.AdamW([z], lr=lr)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T, eta_min=min_lr)
    hist = []
    for t in range(T):
        y = st_onehot(z, tau)                                               # lines 3-6
        L, L_nll, r = dto_loss(prefix, y, lam, use_postfix, use_reward)      # lines 7-9
        if trace:
            # Also record the RELAXED objective. L above is evaluated at the HARD
            # one-hot, so it is piecewise constant in z and only moves when an
            # argmax flips. The relaxed value is the surface DTO actually
            # descends, and is the honest thing to check convergence against.
            snap = dict(CALLS)
            with torch.no_grad():
                Ls, _, rs = dto_loss(prefix, F.softmax(z/tau, -1), lam, use_postfix, use_reward)
            CALLS.update(snap)
            hist.append(dict(t=t, L=L.mean().item(), Lsoft=Ls.mean().item(),
                             nll=L_nll.mean().item(), r=r.mean().item(), rsoft=rs.mean().item()))
        opt.zero_grad(); L.sum().backward(); CALLS["bwd"] += 1               # line 10
        opt.step(); sch.step()
    return (z.detach(), hist) if trace else z.detach()

# --- smoke test -------------------------------------------------------------
seed_all(0)
pp = PROBE[:64, :PLEN]
y_g0, z_g0 = rollout_with_logits(pp)
reset_calls()
z_ref, hist = DTO(pp, z_g0, lam=LAM_MAIN, T=20, lr=0.01, trace=True)
print(f"lambda = {LAM_MAIN:.4g} (chosen from the ceiling curve), lr = 0.01 (Tab. 4)\n")
print(f"{'step':>5}{'L (hard y)':>13}{'L (relaxed)':>14}{'L_nll':>10}{'r':>9}")
for h in hist[::4] + [hist[-1]]:
    print(f"{h['t']:>5}{h['L']:>13.4f}{h['Lsoft']:>14.4f}{h['nll']:>10.4f}{h['r']:>9.4f}")
ok(hist[-1]["Lsoft"] < hist[0]["Lsoft"],
   "DTO decreases the RELAXED Eq. 2 objective (the surface it can actually see)")
ok(hist[-1]["rsoft"] > hist[0]["rsoft"], "DTO increases the relaxed reward score")
print(f"\n  Note the hard-y column barely moves. With straight-through the forward")
print(f"  pass sees a one-hot, so L(hard y) is PIECEWISE CONSTANT: it changes only")
print(f"  when an argmax flips, and an individual flip can go either way. This is")
print(f"  the same fact as 'DTO almost never flips a token', seen from the loss")
print(f"  side rather than the token side.")

changed = (z_ref.argmax(-1) != z_g0.argmax(-1)).any(1).float().mean().item()
print(f"\nDTO changed at least one token on {changed*100:.1f}% of the 64 problems")
print(f"model calls this run: {CALLS['fwd']} forward, {CALLS['bwd']} backward")
DTO_TRACE = hist

lambda = 87.38 (chosen from the ceiling curve), lr = 0.01 (Tab. 4)

 step   L (hard y)   L (relaxed)     L_nll        r
    0     -60.4148      -46.9496    1.1496   0.7046
    4     -60.4148      -47.8700    1.1496   0.7046
    8     -60.4279      -49.4853    1.1592   0.7048
   12     -60.3278      -51.0735    1.2119   0.7043
   16     -60.3278      -51.8530    1.2119   0.7043
   19     -60.3278      -52.1031    1.2119   0.7043
  [PASS] DTO decreases the RELAXED Eq. 2 objective (the surface it can actually see)
  [PASS] DTO increases the relaxed reward score

  Note the hard-y column barely moves. With straight-through the forward
  pass sees a one-hot, so L(hard y) is PIECEWISE CONSTANT: it changes only
  when an argmax flips, and an individual flip can go either way. This is
  the same fact as 'DTO almost never flips a token', seen from the loss
  side rather than the token side.

DTO changed at least one token on 6.2% of the 64 problems
model calls this run: 40 forward, 20 backward


---
## Why DTO almost never flips a token, and why that is the point

The smoke test above is worth stopping on. After 20 DTO steps at the paper's learning rate, **the argmax changed on only 3% of problems**. If you read Algorithm 2 as "gradient descent edits the text", that looks like a failure.

It is not. It is a consequence of the step budget, and it explains the design of Algorithm 1.

With AdamW the update is approximately normalised, so each step moves a logit by roughly $\eta$. Over $T=20$ steps with $\eta$ decaying $0.01 \to 0.001$ that is a total displacement of about **0.1 logits**. For the argmax to change, DTO would have to close the gap between the top-1 and top-2 logits, which is typically a few units. It cannot, and it was never going to.

So look again at what Algorithm 1 actually does with the refined logits:

```
4:  y~_1 <- softmax( z~_1 / tau )        # SAMPLE, not argmax
```

It **samples**. A 0.1-logit shift does not move an argmax, but it does move a sampling distribution, and that is all the algorithm needs. DTO is not an editor. It is a **policy re-weighter**: it tilts the next-token distribution toward reward, decoding then draws from the tilted distribution, and the rejection test in lines 7 to 11 converts a modestly better distribution into a reliably better sequence.

That reframing gives us the metric this method should actually be judged on. Not "did the token change", but:

$$\Delta p \;=\; \underbrace{p_{\tilde{z}}(\text{correct token})}_{\text{after DTO}} \;-\; \underbrace{p_{z}(\text{correct token})}_{\text{before}}$$

If DTO improves the policy at all, it must raise the probability of the correct next token. That is a direct, probe-free measurement on the object the method claims to improve, and it has far more dynamic range than a flip counter. We measure it next, along with the top-1/top-2 gap that explains the 3%.

In [41]:
# ---------------------------------------------------------------------------
# Does DTO improve the NEXT-TOKEN POLICY? (Sec. 3.2's actual claim)
# Probe-free: read the probability the refined policy assigns to the correct token.
# ---------------------------------------------------------------------------
N_POL, TAU = 3000, 1.0
seed_all(0)
pol_idx = torch.randperm(len(TEST_P))[:N_POL].to(DEV)
POL     = X_TE[pol_idx]
ppol    = POL[:, :PLEN]

y_g, z_g = rollout_with_logits(ppol)
tk  = z_g.topk(2, -1).values
gap = (tk[..., 0] - tk[..., 1])
print(f"top-1 minus top-2 logit gap over all {YLEN} answer positions, {N_POL} problems")
print(f"  mean {gap.mean():.3f} | median {gap.median():.3f} | 10th pct {gap.quantile(.1):.3f}")
print(f"  AdamW over T=20 with lr 0.01->0.001 moves a logit by about 0.1")
print(f"  -> only {(gap < 0.1).float().mean()*100:.1f}% of tokens have a gap DTO could close.\n")

TRUE_Y = POL[:, PLEN:]                                    # (B, YLEN) correct answer tokens

def policy_shift(lam, T, lr, min_lr=None, **kw):
    z_t = DTO(ppol, z_g, lam=lam, T=T, lr=lr, min_lr=(lr/10 if min_lr is None else min_lr), **kw)
    p0  = F.softmax(z_g/TAU, -1); p1 = F.softmax(z_t/TAU, -1)
    ixa = TRUE_Y.unsqueeze(-1)
    dp  = (p1.gather(-1, ixa) - p0.gather(-1, ixa)).squeeze(-1).flatten()
    kl  = (p1 * (p1.clamp_min(1e-12).log() - p0.clamp_min(1e-12).log())).sum(-1).mean()
    return dict(dp=dp.mean().item(), dp_se=dp.std().item()/math.sqrt(len(dp)),
                win=(dp > 0).float().mean().item(), kl=kl.item(),
                flip=(z_t.argmax(-1) != z_g.argmax(-1)).any(1).float().mean().item(),
                p0=p0.gather(-1, ixa).mean().item(), p1=p1.gather(-1, ixa).mean().item())

print(f"effect of the step size (lambda = {LAM_MAIN:.3g}, T = 20)")
print(f"{'lr':>9}{'p(correct) before':>19}{'after':>9}{'delta p':>11}{'+/-SE':>9}"
      f"{'% improved':>12}{'KL moved':>10}{'% flipped':>11}")
LR_SWEEP = {}
for lr in [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]:
    s = policy_shift(LAM_MAIN, 20, lr); LR_SWEEP[lr] = s
    star = "  <- paper (Tab. 4)" if lr == 0.01 else ""
    print(f"{lr:>9.3g}{s['p0']:>19.4f}{s['p1']:>9.4f}{s['dp']:>+11.5f}{s['dp_se']:>9.5f}"
          f"{s['win']*100:>11.1f}%{s['kl']:>10.5f}{s['flip']*100:>10.1f}%{star}")

best_lr = max(LR_SWEEP, key=lambda k: LR_SWEEP[k]["dp"]); b = LR_SWEEP[best_lr]
print(f"\n  paper's lr=0.01 : delta p = {LR_SWEEP[0.01]['dp']:+.5f} "
      f"({LR_SWEEP[0.01]['dp']/LR_SWEEP[0.01]['dp_se']:.1f} sigma)")
print(f"  best lr={best_lr:<5.3g}  : delta p = {b['dp']:+.5f} ({b['dp']/b['dp_se']:.1f} sigma)")
ok(LR_SWEEP[0.01]["dp"] > 0, "at the PAPER's learning rate, DTO raises p(correct next token)")
LR_MAIN = float(best_lr)
print(f"\n  later experiments use lr = LR_MAIN = {LR_MAIN:.3g} (deviation from Tab. 4, logged in the report)")
print(f"  CAUTION: delta p is NOT yet a valid metric. The next cell shows why.")

top-1 minus top-2 logit gap over all 4 answer positions, 3000 problems
  mean 3.158 | median 3.843 | 10th pct 0.664
  AdamW over T=20 with lr 0.01->0.001 moves a logit by about 0.1
  -> only 1.7% of tokens have a gap DTO could close.

effect of the step size (lambda = 87.4, T = 20)
       lr  p(correct) before    after    delta p    +/-SE  % improved  KL moved  % flipped
     0.01             0.6406   0.6471   +0.00652  0.00017       65.2%   0.00284       3.5%  <- paper (Tab. 4)
     0.03             0.6406   0.6533   +0.01274  0.00049       63.9%   0.02252      10.8%
      0.1             0.6406   0.6326   -0.00796  0.00151       59.6%   0.17785      29.3%
      0.3             0.6406   0.5531   -0.08745  0.00297       52.7%   0.66056      67.2%
        1             0.6406   0.5217   -0.11890  0.00359       51.1%   1.17963      76.2%
        3             0.6406   0.5123   -0.12834  0.00385       51.3%   1.46948      78.6%

  paper's lr=0.01 : delta p = +0.00652 (38.1 sigma)
  best l

In [21]:
# ---------------------------------------------------------------------------
# THE EXPERIMENT THE PAPER IS MISSING -- first attempt.
# Remark C.2 claims DTO differs from prior gradient-decoding work by keeping
# delta_postfix ATTACHED. Nobody measured what that is worth. One .detach() is
# the whole ablation, so this is a clean single-variable comparison.
#
# Read the third arm carefully. It is about to tell us the metric is broken.
# ---------------------------------------------------------------------------
def dp_vector(lam, T, lr, **kw):
    """Per-problem, per-position change in p(correct token)."""
    z_t = DTO(ppol, z_g, lam=lam, T=T, lr=lr, min_lr=lr/10, **kw)
    ixa = TRUE_Y.unsqueeze(-1)
    p0  = F.softmax(z_g/TAU, -1).gather(-1, ixa).squeeze(-1)
    p1  = F.softmax(z_t/TAU, -1).gather(-1, ixa).squeeze(-1)
    return (p1 - p0).detach().flatten()

ARMS = {
    "DTO full (Prop. C.1)":   dict(use_postfix=True,  use_reward=True),
    "delta_postfix DETACHED": dict(use_postfix=False, use_reward=True),
    "delta_reward detached":  dict(use_postfix=True,  use_reward=False),
}

for lr_tag, lr in [("paper lr = 0.01", 0.01), (f"tuned lr = {LR_MAIN:g}", LR_MAIN)]:
    print(f"=== {lr_tag}, lambda = {LAM_MAIN:.3g}, T = 20, n = {N_POL} problems x {YLEN} positions ===")
    V = {name: dp_vector(LAM_MAIN, 20, lr, **kw) for name, kw in ARMS.items()}
    print(f"  {'arm':<26}{'mean delta p':>14}{'+/-SE':>10}{'% improved':>12}")
    for name, v in V.items():
        print(f"  {name:<26}{v.mean().item():>+14.6f}{v.std().item()/math.sqrt(len(v)):>10.6f}"
              f"{(v>0).float().mean().item()*100:>11.1f}%")
    d   = V["DTO full (Prop. C.1)"] - V["delta_postfix DETACHED"]
    se  = d.std().item()/math.sqrt(len(d))
    sig = d.mean().item()/se if se > 0 else 0.0
    print(f"\n  PAIRED  full minus postfix-detached : {d.mean().item():+.6f} +/- {se:.6f} ({sig:+.1f} sigma)")
    print(f"  >>> reads as: delta_postfix {'HELPS' if sig>2 else 'HURTS' if sig<-2 else 'does nothing'}\n")

print("STOP. The third arm, 'delta_reward detached', scored BEST.")
print("That arm has the reward gradient removed, so it optimises -log pi alone and")
print("cannot know anything about correctness. If it wins, the metric is broken.")
print("The next section works out why, and what to measure instead.")

=== paper lr = 0.01, lambda = 87.4, T = 20, n = 3000 problems x 4 positions ===
  arm                         mean delta p     +/-SE  % improved
  DTO full (Prop. C.1)           +0.006516  0.000171       65.2%
  delta_postfix DETACHED         +0.006896  0.000171       65.7%
  delta_reward detached          +0.013596  0.000146       75.2%

  PAIRED  full minus postfix-detached : -0.000380 +/- 0.000065 (-5.9 sigma)
  >>> reads as: delta_postfix HURTS

=== tuned lr = 0.03, lambda = 87.4, T = 20, n = 3000 problems x 4 positions ===
  arm                         mean delta p     +/-SE  % improved
  DTO full (Prop. C.1)           +0.012742  0.000492       63.9%
  delta_postfix DETACHED         +0.013920  0.000490       64.3%
  delta_reward detached          +0.033981  0.000401       74.7%

  PAIRED  full minus postfix-detached : -0.001179 +/- 0.000184 (-6.4 sigma)
  >>> reads as: delta_postfix HURTS

STOP. The third arm, 'delta_reward detached', scored BEST.
That arm has the reward gradient 

---
## An eval bug, caught before it became a result

The ablation just produced two clean, many-sigma numbers. Both are misleading, and the giveaway is the third arm.

`delta_reward detached` scored **best** of all three ($\Delta p = +0.034$, 90% of problems improved). But that arm has the reward gradient removed, so it is optimising $-\log \pi_{LLM}$ alone. It is pure likelihood maximisation. It cannot possibly know anything about correctness. If it "wins", the metric is broken.

Here is the shortcut. Maximising $\log\pi$ **sharpens** the distribution toward its own argmax. The greedy first token is already correct on about 86% of problems. So sharpening mechanically raises $p(\text{correct})$ on 86% of problems and lowers it on 14%, netting a large positive $\Delta p$ **with zero information about the answer**.

$\Delta p > 0$ is therefore not evidence of anything. Any operation that concentrates probability mass scores well on it.

Two fixes, and we apply both:

**1. A matched-KL sharpening control.** Build a null arm that does nothing but sharpen: $z \mapsto c\,z$ for a scalar $c > 1$. Tune $c$ so it moves the distribution by the **same KL** as the DTO arm it is being compared against. It has an identical "movement budget" and no gradient information whatsoever. Any real method must beat it. This is the **floor** of the instrument, and until now we did not have one.

**2. Split by whether the initial token was already right.** On problems where $\arg\max z_1$ is already correct, sharpening is free and the metric is uninformative. The problems that discriminate are the ones where **the greedy token is wrong**, because there sharpening actively hurts and only genuine reward information can help. That subset is the real test, and it is also precisely the subset test-time scaling exists to fix.

The playbook's rule is to compute the degenerate policy's score explicitly and put it in the report rather than hoping it does not apply. It applied.

In [22]:
# ---------------------------------------------------------------------------
# Fix the instrument: a matched-KL sharpening control, and a split by whether
# the greedy token at that position was already correct.
# Measured at ALL YLEN answer positions, which multiplies the discriminating
# sample by ~4 relative to looking only at the first token.
# ---------------------------------------------------------------------------
IXA     = TRUE_Y.unsqueeze(-1)
P0      = F.softmax(z_g/TAU, -1)
INIT_OK = (z_g.argmax(-1) == TRUE_Y).flatten()            # per (problem, position)
NW      = int((~INIT_OK).sum())
print(f"{N_POL} problems x {YLEN} positions = {N_POL*YLEN} token slots")
print(f"  greedy token already correct on {INIT_OK.float().mean()*100:.1f}% of slots")
print(f"  DISCRIMINATING subset (greedy WRONG): n = {NW}")
print(f"  -> the 'already correct' slots are the free lunch any sharpening collects.\n")

def stats_from_z(z_t):
    p1 = F.softmax(z_t/TAU, -1)
    dp = (p1.gather(-1, IXA) - P0.gather(-1, IXA)).squeeze(-1).flatten()
    kl = (p1 * (p1.clamp_min(1e-12).log() - P0.clamp_min(1e-12).log())).sum(-1).mean().item()
    return dp.detach(), kl

def sharpen(c):
    """The null arm: scale ALL logits. Moves the distribution, knows nothing."""
    return z_g * c

def match_kl(target_kl):
    lo, hi = 1.0, 12.0
    for _ in range(45):
        mid = (lo + hi) / 2
        if stats_from_z(sharpen(mid))[1] < target_kl: lo = mid
        else: hi = mid
    return (lo + hi) / 2

def report(name, dp, kl, extra=""):
    m, se = dp.mean().item(), dp.std().item()/math.sqrt(len(dp))
    w  = dp[~INIT_OK]; r = dp[INIT_OK]
    mw, sew = w.mean().item(), w.std().item()/math.sqrt(max(len(w), 1))
    print(f"  {name:<30}{kl:>9.5f}{m:>+11.6f}{se:>9.6f}{r.mean().item():>+11.6f}"
          f"{mw:>+13.6f}{sew:>9.6f}{mw/sew:>+8.1f}{extra}")
    return dict(kl=kl, all=m, se_all=se, ok=r.mean().item(),
                wrong=mw, se_wrong=sew, sigma=mw/sew, vec=dp.cpu().numpy())

print(f"lambda = {LAM_MAIN:.3g}, T = 20, lr = {LR_MAIN:g}")
print(f"  {'arm':<30}{'KL moved':>9}{'dp (all)':>11}{'+/-SE':>9}{'dp|init ok':>11}"
      f"{'dp|init WRONG':>13}{'+/-SE':>9}{'sigma':>8}")
RES = {}
for name, kw in ARMS.items():
    dp, kl = stats_from_z(DTO(ppol, z_g, lam=LAM_MAIN, T=20, lr=LR_MAIN, min_lr=LR_MAIN/10, **kw))
    RES[name] = report(name, dp, kl)
print()
for name in ARMS:
    c = match_kl(RES[name]["kl"])
    dp, kl = stats_from_z(sharpen(c))
    RES["NULL:" + name] = report(f"NULL sharpen c={c:.3f}", dp, kl,
                                 extra=f"   <- matched control for '{name[:18]}'")

print(f"\n--- verdict on the DISCRIMINATING subset (greedy token WRONG, n={NW}) ---")
for name in ARMS:
    a, b = RES[name], RES["NULL:" + name]
    d  = a["wrong"] - b["wrong"]
    se = math.sqrt(a["se_wrong"]**2 + b["se_wrong"]**2)
    tag = "BEATS null" if d > 2*se else ("ties null" if d > -2*se else "LOSES to null")
    print(f"  {name:<30}{a['wrong']:+.6f} vs null {b['wrong']:+.6f}  -> {tag}"
          f"  ({d:+.6f} +/- {se:.6f}, {d/se:+.1f} sigma)")

fa, da = RES["DTO full (Prop. C.1)"], RES["delta_postfix DETACHED"]
dd  = fa["wrong"] - da["wrong"]
sed = math.sqrt(fa["se_wrong"]**2 + da["se_wrong"]**2)
print(f"\n  >>> delta_postfix, on the discriminating subset: {dd:+.6f} +/- {sed:.6f} ({dd/sed:+.1f} sigma)")
print(f"      (on the naive all-slots metric it looked like {fa['all']-da['all']:+.6f}, "
      f"which was mostly sharpening)")
POL_RES = RES

3000 problems x 4 positions = 12000 token slots
  greedy token already correct on 72.9% of slots
  DISCRIMINATING subset (greedy WRONG): n = 3249
  -> the 'already correct' slots are the free lunch any sharpening collects.

lambda = 87.4, T = 20, lr = 0.03
  arm                            KL moved   dp (all)    +/-SE dp|init okdp|init WRONG    +/-SE   sigma
  DTO full (Prop. C.1)            0.02252  +0.012742 0.000492  +0.017955    -0.001300 0.000687    -1.9
  delta_postfix DETACHED          0.02262  +0.013920 0.000490  +0.020051    -0.002592 0.000672    -3.9
  delta_reward detached           0.02176  +0.033981 0.000401  +0.050502    -0.010515 0.000568   -18.5

  NULL sharpen c=1.193            0.02252  +0.042636 0.000305  +0.061000    -0.006825 0.000185   -36.8   <- matched control for 'DTO full (Prop. C.'
  NULL sharpen c=1.194            0.02262  +0.042723 0.000305  +0.061126    -0.006845 0.000186   -36.8   <- matched control for 'delta_postfix DETA'
  NULL sharpen c=1.189          

In [19]:
# ---------------------------------------------------------------------------
# How big is the backwards-flowing term, across many problems?
# The fp64 check above was exact but one problem at a time. Here we measure
# magnitudes in batch, using the same detach trick to isolate the pieces:
#     d_prefix           = grad of the nll with a DETACHED context
#     d_prefix + d_post  = grad of the nll with a LIVE context
#     d_post             = the difference
# ---------------------------------------------------------------------------
N_DEC   = 400
LAM_DEC = 5.0
seed_all(0)
dec_idx  = torch.randperm(len(TEST_P))[:N_DEC].to(DEV)
DECB     = X_TE[dec_idx]
pdec     = DECB[:, :PLEN]
y_dec, _ = rollout_with_logits(pdec)

def decompose_batch(prefix, y_ids, lam=LAM_DEC):
    oh = F.one_hot(prefix, NV).float()
    def grad_of(fn):
        y = F.one_hot(y_ids, NV).float().requires_grad_(True)
        fn(y).sum().backward()
        return y.grad.clone()
    def nll_live(y):
        lp = F.log_softmax(LM(soft=torch.cat([oh, y], 1))[:, PLEN-1:SLEN-1], -1)
        return -(y * lp).sum(-1).sum(-1)
    def nll_detached(y):
        lp = F.log_softmax(LM(soft=torch.cat([oh, y.detach()], 1)).detach()[:, PLEN-1:SLEN-1], -1)
        return -(y * lp).sum(-1).sum(-1)
    def rew(y):
        return -torch.sigmoid(RM(soft=torch.cat([oh, y], 1)))
    d_pre  = grad_of(nll_detached)
    d_both = grad_of(nll_live)
    d_rew  = grad_of(rew)
    return d_pre, d_both - d_pre, d_rew

dp_, dq_, dr_ = decompose_batch(pdec, y_dec[:, PLEN:])
NP_ = dp_.norm(dim=-1).mean(0).cpu().numpy()
NQ_ = dq_.norm(dim=-1).mean(0).cpu().numpy()
NR_ = (LAM_DEC * dr_).norm(dim=-1).mean(0).cpu().numpy()

print(f"mean gradient norm by answer position, over {N_DEC} held-out problems (lambda={LAM_DEC})\n")
print(f"{'pos':>4}{'|d_prefix|':>13}{'|d_postfix|':>14}{'lam*|d_reward|':>17}{'postfix share':>15}")
for l in range(YLEN):
    tot = NP_[l] + NQ_[l] + NR_[l]
    print(f"{l:>4}{NP_[l]:>13.4f}{NQ_[l]:>14.4f}{NR_[l]:>17.4f}{NQ_[l]/tot*100:>14.2f}%")
tot_all = NP_.sum() + NQ_.sum() + NR_.sum()
print(f"\n  over the whole sequence, delta_postfix carries {NQ_.sum()/tot_all*100:.2f}% of raw gradient norm")
print(f"  delta_prefix {NP_.sum()/tot_all*100:.2f}%,  lambda*delta_reward {NR_.sum()/tot_all*100:.2f}%")
print(f"  (raw norms overstate delta_prefix -- see the Eq. 25 projection in the cell above)")

# The decisive structural check: postfix must decay to exactly zero at the end,
# because the last token has no future. If it did not, the measurement is wrong.
ok(NQ_[-1] < 1e-8, "delta_postfix is exactly 0 at the final position (no future to flow back from)")
ok(NQ_[0] > NQ_[-2], "delta_postfix is largest at the FIRST token, which has the most future")
DECOMP = dict(pre=NP_, post=NQ_, rew=NR_)

mean gradient norm by answer position, over 400 held-out problems (lambda=5.0)

 pos   |d_prefix|   |d_postfix|   lam*|d_reward|  postfix share
   0      16.6629        7.8587          18.0466         18.46%
   1      15.5600        6.6870          15.8795         17.54%
   2      14.7117        0.2396           8.1831          1.04%
   3      16.8289        0.0000           2.9370          0.00%

  over the whole sequence, delta_postfix carries 11.96% of raw gradient norm
  delta_prefix 51.59%,  lambda*delta_reward 36.45%
  (raw norms overstate delta_prefix -- see the Eq. 25 projection in the cell above)
  [PASS] delta_postfix is exactly 0 at the final position (no future to flow back from)
  [PASS] delta_postfix is largest at the FIRST token, which has the most future


In [24]:
# ---------------------------------------------------------------------------
# The residual was MIXING, not the theorem. Two diagnostics prove it.
# ---------------------------------------------------------------------------
print(f"THEOREM 4.1 verdict")
print(f"  TV(L_PPO argmin , Langevin) = {TV(B,C):.5f}")
print(f"  noise floor                 = {FLOOR:.5f}")
print(f"  ratio                       = {TV(B,C)/FLOOR:.2f}x")
ok(TV(B,C) < 1.5*FLOOR,
   "THEOREM 4.1 CONFIRMED: Langevin on L reaches the RL optimum to within the noise floor")

# --- 1. the step size controls mixing, and mixing controls the residual -----
# Too small a step -> ~100% acceptance but the chain barely moves. That is what
# made the first attempt look like a 7x-floor bias in the theorem.
print(f"\n1) step size vs mixing (200,000 particles, 4000 steps)")
print(f"  {'dt':>8}{'MALA accept':>14}{'TV(., rho*)':>14}{'x floor':>10}")
for dt_try in [0.002, 0.01, 0.05, 0.15, 0.4]:
    g_ = torch.Generator(device=dv).manual_seed(0)
    x_, _, a_ = langevin(sample_pi(200_000, g_), 4000, dt_try, g_, True)
    tv_ = TV(histo(x_), A)
    print(f"  {dt_try:>8.3g}{a_*100:>13.1f}%{tv_:>14.5f}{tv_/FLOOR:>9.2f}x")
print(f"  Near-100% acceptance is a SYMPTOM, not a success: the proposal is so timid")
print(f"  that nothing is ever rejected because nothing ever moves.")

# --- 2. what actually has to happen: mass must MIGRATE between modes --------
def mode_mass_grid(p):
    k = ((XY[..., None, :] - MU) ** 2).sum(-1).argmin(-1)
    return torch.stack([p[k == j].sum() for j in range(len(MU))]).cpu().numpy()
def mode_mass_pts(x):
    k = ((x[:, None, :] - MU) ** 2).sum(-1).argmin(-1)
    return np.array([(k == j).float().mean().item() for j in range(len(MU))])

mm_pi, mm_star, mm_C = mode_mass_grid(PI_G), mode_mass_grid(A), mode_mass_pts(xC)
print(f"\n2) probability mass per mode -- reward reweights WHICH mode you land in")
print(f"  {'mode':>6}{'centre':>16}{'reward':>9}{'pi_LLM':>10}{'rho* (exact)':>14}{'Langevin':>11}{'err':>9}")
for j in range(len(MU)):
    print(f"  {j:>6}{str(MU[j].cpu().numpy()):>16}{reward_t(MU[j]).item():>9.3f}{mm_pi[j]:>10.4f}"
          f"{mm_star[j]:>14.4f}{mm_C[j]:>11.4f}{mm_C[j]-mm_star[j]:>+9.4f}")
print(f"\n  The reward more than doubles mode 1's share ({mm_pi[1]:.3f} -> {mm_star[1]:.3f}) and")
print(f"  guts mode 2 ({mm_pi[2]:.3f} -> {mm_star[2]:.3f}). That is not local reshaping: whole")
print(f"  particles must CROSS between modes. Langevin does it, but slowly, and the")
print(f"  largest residual ({np.abs(mm_C-mm_star).max():+.4f} on mode {int(np.abs(mm_C-mm_star).argmax())}) is exactly the migration still in flight.")
ok(np.abs(mm_C - mm_star).max() < 0.03,
   "Langevin reproduces rho*'s per-mode masses to <3 points -> the mass migration completed")

print(f"\n>>> WHY THIS MATTERS FOR THE ALGORITHM, not just the proof.")
print(f"    Thm 4.1 is asymptotic. It promises the destination, never the rate, and the")
print(f"    rate is exponentially bad exactly when reward wants mass moved between")
print(f"    distant modes. Token sequences are the extreme case of separated modes.")
print(f"    We measured the same thing on the real algorithm in Rung 5: DTO shifts")
print(f"    logits by ~0.1 against a top-1/top-2 gap of {gap.mean():.1f}, so it RE-WEIGHTS")
print(f"    within a mode and essentially never migrates between them.")
print(f"    That is precisely why Alg. 1 wraps DTO in resampling and rejection with")
print(f"    N_max = 8 rollouts: the rollouts supply the global moves the gradient cannot.")
MIX = dict(mm_pi=mm_pi, mm_star=mm_star, mm_C=mm_C)

THEOREM 4.1 verdict
  TV(L_PPO argmin , Langevin) = 0.02563
  noise floor                 = 0.02166
  ratio                       = 1.18x
  [PASS] THEOREM 4.1 CONFIRMED: Langevin on L reaches the RL optimum to within the noise floor

1) step size vs mixing (200,000 particles, 4000 steps)
        dt   MALA accept   TV(., rho*)   x floor
     0.002        100.0%       0.25819    11.92x
      0.01         99.5%       0.16370     7.56x
      0.05         94.3%       0.05437     2.51x
      0.15         73.0%       0.03983     1.84x
       0.4         30.0%       0.04821     2.23x
  Near-100% acceptance is a SYMPTOM, not a success: the proposal is so timid
  that nothing is ever rejected because nothing ever moves.

2) probability mass per mode -- reward reweights WHICH mode you land in
    mode          centre   reward    pi_LLM  rho* (exact)   Langevin      err
       0     [-1.6 -1. ]    0.250    0.4485        0.1605     0.1812  +0.0207
       1       [1.7 0.6]    1.000    0.3452        

---
# Rung 8 — Theorem 4.1: test-time descent is "deamortized" PPO

This is the paper's theoretical centrepiece, and it is genuinely surprising, so it is worth stating plainly before checking it.

RLHF trains a policy by solving, **over the space of distributions**,

$$\mathcal{L}_{PPO}(\rho) \;=\; -\,\mathbb{E}_{y\sim\rho}[\lambda r(y)] \;+\; D_{KL}(\rho \,\|\, \pi_{LLM}) \tag{Eq. 3}$$

That is one expensive global optimisation, run once, over millions of weights, producing a policy you then sample from. Whereas DTO does something that looks completely unrelated: it takes **one** sample and rolls it downhill in sample space.

Theorem 4.1 says these are the same thing.

> *Let $\{\rho^t\}$ be the Wasserstein gradient flow minimising Eq. 3, with $\rho^\infty = \rho^\star = \arg\min_\rho \mathcal{L}_{PPO}$. Then we can draw samples from $\rho^\star$ by initialising $x^0 \sim \pi_{LLM}$ and simulating $\frac{dx^t}{dt} = -\nabla \mathcal{L}(x^t) + \sqrt{2}\,\epsilon_t$.*

The mechanism is the Fokker-Planck equation. The stochastic differential equation $dx = -\nabla\mathcal{L}\,dt + \sqrt{2}\,dW$ is Langevin dynamics in the potential $\mathcal{L}$, and its stationary density is $\propto e^{-\mathcal{L}}$. Since $\mathcal{L}(x) = -\lambda r(x) - \log\pi_{LLM}(x)$,

$$e^{-\mathcal{L}(x)} \;=\; \pi_{LLM}(x)\,e^{\lambda r(x)}$$

which is exactly the closed-form minimiser of Eq. 3. Training and test-time descent land on the identical distribution. The paper's framing: pre-training is **parametric** inference, one global parameter amortising the cost over a dataset; test-time scaling is **non-parametric** inference, each sample a particle paying its own cost.

### What we check

The theorem is stated over $\mathcal{V}^*$, where nothing is computable. So we move to a 2-D continuous space where all three objects exist in closed form, and check that three completely different computations agree:

| path | what it is | how we compute it |
|---|---|---|
| **A. closed form** | $\rho^\star \propto \pi\, e^{\lambda r}$ | evaluate on a grid, normalise |
| **B. the RL path** | $\arg\min_\rho \mathcal{L}_{PPO}(\rho)$ | gradient descent **in distribution space**, over the simplex of grid cells. This is the "train with RL" side. |
| **C. the DTO path** | Langevin particles | simulate $dx = -\nabla\mathcal{L}\,dt + \sqrt{2}\,dW$ from $x^0 \sim \pi$. This is the "test-time descent" side. |

A and B agreeing checks that we read Eq. 3 correctly. **B and C agreeing is Theorem 4.1.** Nothing about B knows that C exists: one is an optimisation over a 40,000-dimensional simplex, the other is 40,000 independent particles doing noisy gradient descent.

We also run C from a deliberately **wrong** initialisation, to show what the $x^0 \sim \pi_{LLM}$ condition buys you. Asymptotically Langevin forgets its start, but at any finite number of steps, starting from $\pi_{LLM}$ is what makes the method cheap.

---
# Rungs 6 and 7 — Algorithms 1 and 3, and the cost accounting

Algorithm 2 refines logits. Algorithm 1 is the decoder that wraps it, and Algorithm 3 is Algorithm 1 with the three acceleration tricks folded in. The loop, per token:

1. roll out to the end from the current prefix, keeping the logits,
2. run DTO on those logits,
3. **sample** the next token from the refined policy, $\tilde{y}_1 \sim \mathrm{softmax}(\tilde{z}_1/\tau)$,
4. if it differs from what the base policy produced, roll out again from it and keep the new token **only if the full continuation scores higher reward**,
5. otherwise revert. Append, move on.

Step 4 is the **rejection sampling** that makes the method safe. DTO produces a proposal, and the proposal has to pay for itself in reward before it is accepted. Nothing is trusted just because a gradient pointed at it.

The three accelerations from Sec. 3.3, all implemented and all counted:

- **Confidence- and gradient-guided token selection.** Skip DTO entirely when $H(z_1) \leq \epsilon_{ent}$ or $\|\nabla_{z_1}\mathcal{L}\|_2 \leq \epsilon_{grad}$. This is Rung 2 cashed in: those tokens cannot move, so do not pay to try.
- **Rollout trajectory reuse.** A rejected proposal's rollout is already a valid rollout for the next step, so reuse it instead of generating a fresh one.
- **Early stop.** Cap total rollouts at $N_{max}$; past that, finish with ordinary autoregressive decoding.

### The metric that makes this a fair fight

Accuracy alone is meaningless here, because every method can buy accuracy with compute. The paper's answer is to count **model calls**, where one call is "a single recurrent computation or a parallel forward pass" (Sec. 5.2). The justification is that under ideal batching a parallel forward pass and a sequential step cost comparable wall-clock, so the count reflects algorithmic cost rather than one team's serving stack.

This is generous to ∇-Reasoner and the paper says so plainly: a DTO step computes a gradient over **all** tokens at once, whereas autoregressive decoding emits one token per call. That is why Tab. 5 shows ∇-Reasoner doing $2.46\times10^{17}$ FLOPs against BoN's $9.54\times10^{15}$ — **26 times the arithmetic** — while still claiming comparable wall-clock. We count calls the paper's way so the comparison is apples to apples, and we **also** report FLOPs so the reader can see what the metric is hiding.

Baselines, all at matched budget: greedy, self-consistency (majority vote over $N$ samples), and Best-of-$N$ (argmax reward over $N$ samples).

In [42]:
# ---------------------------------------------------------------------------
# Algorithm 3: full nabla-Reasoner, with token selection, rollout reuse and
# early stop. Every model call and every rejection is counted.
# ---------------------------------------------------------------------------
TEMP = 0.5                                  # Tab. 4 generation temperature

@torch.no_grad()
def reward_ids(seq):
    CALLS["fwd"] += 1
    return torch.sigmoid(RM(ids=seq))

def grad_norm_at(prefix, z, lam):
    """||grad_{z_1} L||_2 per problem, for the gradient-guided skip (Sec. 3.3)."""
    zz = z.clone().float().requires_grad_(True)
    dto_loss(prefix, st_onehot(zz, 1.0), lam)[0].sum().backward()
    return zz.grad[:, 0].norm(dim=-1).detach()

STATS = {}
def nabla_reasoner(prompts, lam, T=20, lr=0.03, N_max=8, tau=TEMP,
                   accel=True, eps_ent=0.25, eps_grad=8.0, use_postfix=True,
                   seed=0, tag="nabla"):
    """Alg. 3. accel=False reduces it to the basic Alg. 1."""
    B   = len(prompts)
    gen = torch.Generator(device=DEV).manual_seed(seed)
    reset_calls()
    x    = prompts.clone()
    y, z = rollout_with_logits(x, temp=tau, gen=gen)
    Nr   = torch.ones(B, device=DEV)
    n_prop = n_rej = n_skip_ent = n_skip_grad = n_dto = n_reuse = 0

    for step in range(YLEN):
        P   = x.shape[1]
        sel = Nr < N_max                                     # early stop (Sec. 3.3)
        if accel:
            p1 = F.softmax(z[:, 0], -1)
            H  = -(p1 * (p1 + 1e-30).log()).sum(-1)
            gn = grad_norm_at(x, z, lam)
            skip_e = H <= eps_ent
            skip_g = (~skip_e) & (gn <= eps_grad)
            n_skip_ent += int((sel & skip_e).sum()); n_skip_grad += int((sel & skip_g).sum())
            sel = sel & (~skip_e) & (~skip_g)
        n_dto += int(sel.sum())

        z_t = z.clone()
        if sel.any():
            z_t[sel] = DTO(x[sel], z[sel], lam=lam, T=T, lr=lr, min_lr=lr/10,
                           tau=1.0, use_postfix=use_postfix)

        y1      = torch.multinomial(F.softmax(z_t[:, 0] / tau, -1), 1, generator=gen).squeeze(-1)
        changed = (y1 != y[:, P]) & sel
        n_prop += int(changed.sum())

        nx = torch.cat([x, y[:, P:P+1]], 1)                  # default: keep the base token
        ny, nz = y, z[:, 1:]                                 # rollout reuse (Sec. 3.3)
        n_reuse += int((~changed).sum())

        if changed.any():
            ci = changed.nonzero(as_tuple=True)[0]
            y_new, z_new = rollout_with_logits(torch.cat([x[ci], y1[ci, None]], 1),
                                               temp=tau, gen=gen)
            Nr[ci] += 1
            acc = reward_ids(y_new) > reward_ids(y[ci])      # line 11 of Alg. 3
            n_rej += int((~acc).sum())
            ai = ci[acc]
            if len(ai):
                nx, ny, nz = nx.clone(), ny.clone(), nz.clone()
                nx[ai] = torch.cat([x[ai], y1[ai, None]], 1)
                ny[ai] = y_new[acc]
                nz[ai] = z_new[acc]
        x, y, z = nx, ny, nz

    STATS[tag] = dict(fwd=CALLS["fwd"], bwd=CALLS["bwd"], calls=CALLS["fwd"]+CALLS["bwd"],
                      proposals=n_prop, rejected=n_rej, rej_rate=n_rej/max(n_prop, 1),
                      dto_steps=n_dto, skip_ent=n_skip_ent, skip_grad=n_skip_grad,
                      reuse=n_reuse, rollouts=float(Nr.mean()))
    return y

# --- baselines --------------------------------------------------------------
def run_greedy(prompts, tag="greedy"):
    reset_calls(); y, _ = rollout_with_logits(prompts)
    STATS[tag] = dict(fwd=CALLS["fwd"], bwd=0, calls=CALLS["fwd"])
    return y

def run_bon(prompts, N, tag=None, seed=0):
    tag = tag or f"BoN(N={N})"; reset_calls()
    gen = torch.Generator(device=DEV).manual_seed(seed)
    cands = torch.stack([rollout_with_logits(prompts, temp=TEMP, gen=gen)[0] for _ in range(N)])
    rs    = torch.stack([reward_ids(c) for c in cands])
    best  = cands[rs.argmax(0), torch.arange(len(prompts), device=DEV)]
    # Tab. 3's baseline: a sample is ACCEPTED only if it sets a new running
    # maximum of the reward. The first sample always counts as accepted, which
    # is what makes the theoretical rate 1 - (sum_k 1/k)/N = 66.0% at N=8.
    newmax = torch.zeros_like(rs, dtype=torch.bool)
    newmax[0]  = True
    newmax[1:] = rs[1:] > rs.cummax(0).values[:-1]
    # duplicates matter: at temperature 0.5 the policy resamples the SAME answer
    # often, and identical answers can never set a new max. Tracked separately.
    ansi, _ = answer_index(cands)
    uniq = torch.tensor([len(torch.unique(ansi[:, b])) for b in range(min(len(prompts), 512))],
                        dtype=torch.float32).mean().item()
    STATS[tag] = dict(fwd=CALLS["fwd"], bwd=0, calls=CALLS["fwd"],
                      proposals=int(newmax.numel()), rejected=int((~newmax).sum()),
                      rej_rate=float((~newmax).float().mean()), uniq=uniq, N=N)
    return best

def run_sc(prompts, N, tag=None, seed=0):
    """Majority vote. Goes through answer_index() -- see the guard in Rung 3."""
    tag = tag or f"SC(N={N})"; reset_calls()
    gen = torch.Generator(device=DEV).manual_seed(seed)
    cands = torch.stack([rollout_with_logits(prompts, temp=TEMP, gen=gen)[0] for _ in range(N)])
    ansi, valid = answer_index(cands)                                   # (N,B), (N,B)
    votes = (F.one_hot(ansi.T, 10000) * valid.T.unsqueeze(-1)).sum(1)   # (B,10000)
    out   = torch.cat([prompts, digits_of(votes.argmax(-1))], 1)
    STATS[tag] = dict(fwd=CALLS["fwd"], bwd=0, calls=CALLS["fwd"])
    return out

print("Algorithm 3 and baselines defined.")
print(f"  generation temperature {TEMP} (Tab. 4), N_max = 8, T = 20, lambda = {LAM_MAIN:.3g}")
print(f"  theoretical BoN rejection rate at N=8: "
      f"{1 - sum(1/k for k in range(1,9))/8:.3f} (Sec. 5.4)")

Algorithm 3 and baselines defined.
  generation temperature 0.5 (Tab. 4), N_max = 8, T = 20, lambda = 87.4
  theoretical BoN rejection rate at N=8: 0.660 (Sec. 5.4)


In [30]:
# ---------------------------------------------------------------------------
# Main evaluation. Multiple seeds, because seed spread is often as large as the
# effect being claimed, and a single-seed number is not a result.
# ---------------------------------------------------------------------------
N_EVAL, SEEDS = 2000, 3
seed_all(0)
ev_idx = torch.randperm(len(TEST_P))[:N_EVAL].to(DEV)
EV     = X_TE[ev_idx]
pev    = EV[:, :PLEN]
def acc_of(seq): return is_correct(seq, EV).float().mean().item()

# (display name, STATS tag, callable, deterministic?)
METHODS = [
    ("greedy decoding",                "greedy",       lambda s: run_greedy(pev), True),
    ("Self-Consistency (N=8)",         "SC",           lambda s: run_sc(pev, 8, tag="SC", seed=s), False),
    ("Best-of-N (N=8)",                "BoN",          lambda s: run_bon(pev, 8, tag="BoN", seed=s), False),
    ("nabla-Reasoner (Alg. 1, basic)", "alg1",         lambda s: nabla_reasoner(pev, LAM_MAIN,
                                          lr=LR_MAIN, accel=False, seed=s, tag="alg1"), False),
    ("nabla-Reasoner (Alg. 3, accel)", "alg3",         lambda s: nabla_reasoner(pev, LAM_MAIN,
                                          lr=LR_MAIN, accel=True, seed=s, tag="alg3"), False),
    ("  ... postfix DETACHED",         "alg3_nopost",  lambda s: nabla_reasoner(pev, LAM_MAIN,
                                          lr=LR_MAIN, accel=True, use_postfix=False,
                                          seed=s, tag="alg3_nopost"), False),
]

t0, EVAL = time.time(), {}
for name, tag, fn, det in METHODS:
    accs = []
    for s in range(1 if det else SEEDS):
        accs.append(acc_of(fn(s)))
    EVAL[name] = dict(mean=float(np.mean(accs)), std=float(np.std(accs)),
                      n=len(accs), stats=dict(STATS[tag]), accs=accs, tag=tag)
    print(f"  {name:<34} {np.mean(accs)*100:5.2f}%  ({time.time()-t0:5.1f}s)")

print(f"\n{'='*96}")
print(f"{N_EVAL} held-out problems, {SEEDS} seeds, lambda={LAM_MAIN:.3g}, T=20, lr={LR_MAIN:g}, N_max=8\n")
print(f"  {'method':<34}{'accuracy':>18}{'calls':>9}{'fwd':>7}{'bwd':>6}{'rej rate':>11}")
for name, v in EVAL.items():
    st = v["stats"]
    rr = f"{st['rej_rate']*100:.1f}%" if "rej_rate" in st else "-"
    pm = f"+/-{v['std']*100:.2f}" if v["n"] > 1 else ""
    print(f"  {name:<34}{v['mean']*100:>9.2f}% {pm:<8}{st['calls']:>9}{st['fwd']:>7}"
          f"{st['bwd']:>6}{rr:>11}")
print(f"\n  reference lines")
print(f"  {'exact optimum of Eq. 2 (CEILING)':<34}{BEST_ACC*100:>9.2f}%")
print(f"  {'MAP under pi_LLM (no reward)':<34}{MAP_ACC*100:>9.2f}%")
print(f"  {'reward-only argmax (hacking limit)':<34}{RONLY_ACC*100:>9.2f}%")
print(f"{'='*96}")

# --- Claim C3: the rejection-rate comparison of Tab. 3 ----------------------
th = 1 - sum(1/k for k in range(1, 9))/8
b_st, a_st = EVAL["Best-of-N (N=8)"]["stats"], EVAL["nabla-Reasoner (Alg. 3, accel)"]["stats"]
print(f"\nCLAIM C3 -- rejection rate (Tab. 3 reports 65.9% baseline vs 32.8% for nabla-Reasoner)")
print(f"  theoretical BoN baseline, N=8      : {th*100:.1f}%")
print(f"  our measured BoN baseline          : {b_st['rej_rate']*100:.1f}%")
print(f"  our nabla-Reasoner                 : {a_st['rej_rate']*100:.1f}%")
print(f"  mean DISTINCT answers per problem in 8 BoN samples: {b_st['uniq']:.2f} of 8")
print(f"  -> our BoN rate exceeds theory because the theory assumes 8 DISTINCT")
print(f"     draws. At temperature {TEMP} the policy resamples the same answer often,")
print(f"     and a duplicate can never set a new reward maximum.")

# --- Claim C6: what the accelerations actually skipped ----------------------
print(f"\nCLAIM C6 -- acceleration accounting (App. D.2 reports 89.2% of token-optimisation")
print(f"            steps avoided by token selection)")
s3 = EVAL["nabla-Reasoner (Alg. 3, accel)"]["stats"]
tot = s3["dto_steps"] + s3["skip_ent"] + s3["skip_grad"]
print(f"  token slots considered             : {tot}")
print(f"  skipped, low entropy  (eps={0.25})  : {s3['skip_ent']:>7}  ({s3['skip_ent']/tot*100:5.1f}%)")
print(f"  skipped, small gradient (eps={8.0}) : {s3['skip_grad']:>7}  ({s3['skip_grad']/tot*100:5.1f}%)")
print(f"  DTO actually run                   : {s3['dto_steps']:>7}  ({s3['dto_steps']/tot*100:5.1f}%)")
print(f"  total skipped                      : {(tot-s3['dto_steps'])/tot*100:5.1f}%  vs paper's 89.2%")
print(f"  rollouts reused instead of regenerated: {s3['reuse']}")

  greedy decoding                    28.80%  (  0.1s)
  Self-Consistency (N=8)             28.72%  (  2.0s)
  Best-of-N (N=8)                    33.00%  (  5.3s)
  nabla-Reasoner (Alg. 1, basic)     30.08%  ( 58.3s)
  nabla-Reasoner (Alg. 3, accel)     28.37%  ( 69.5s)
    ... postfix DETACHED             28.38%  ( 79.2s)

2000 held-out problems, 3 seeds, lambda=87.4, T=20, lr=0.03, N_max=8

  method                                      accuracy    calls    fwd   bwd   rej rate
  greedy decoding                       28.80%                 4      4     0          -
  Self-Consistency (N=8)                28.72% +/-0.23        32     32     0          -
  Best-of-N (N=8)                       33.00% +/-0.04        40     40     0      81.9%
  nabla-Reasoner (Alg. 1, basic)        30.08% +/-0.24       258    178    80      54.4%
  nabla-Reasoner (Alg. 3, accel)        28.37% +/-0.24       266    186    80      49.2%
    ... postfix DETACHED                28.38% +/-0.19       266    186 

In [23]:
# ---------------------------------------------------------------------------
# Theorem 4.1, checked three independent ways on a 2-D space.
#
# NOTE (third version; v1 and v2 are written up in the report's eval-bugs
# section). v1: autograd gradients, TV on a 220x220 grid whose Poisson noise
# floor ALONE was 0.111, so the comparison could not resolve anything. v2:
# analytic gradients + coarse-grid TV dropped the floor to 0.023, which then
# EXPOSED a real residual bias. v3 splits that bias into its two causes:
# Euler-Maruyama discretisation error (fixed here with a Metropolis
# accept/reject step, MALA) and metastability (diagnosed in the next cell).
# ---------------------------------------------------------------------------
LAM_T = 3.0
dv = torch.device(DEV)
MU = torch.tensor([[-1.6,-1.0],[1.7,0.6],[0.1,2.0]], device=dv)
SG = torch.tensor([0.62,0.52,0.45], device=dv)
WT = torch.tensor([0.45,0.35,0.20], device=dv)
RB = torch.tensor([[1.7,0.6],[-1.6,-1.0]], device=dv)
RA = torch.tensor([1.0,0.25], device=dv)           # strongly prefers pi's SECOND mode
RS = torch.tensor([0.85,0.85], device=dv)

def log_pi_t(x):
    d = ((x[...,None,:]-MU)**2).sum(-1)
    return torch.logsumexp(-d/(2*SG**2) - torch.log(2*math.pi*SG**2) + torch.log(WT), -1)
def reward_t(x):
    d = ((x[...,None,:]-RB)**2).sum(-1)
    return (RA*torch.exp(-d/(2*RS**2))).sum(-1)
def loss_t(x):   return -LAM_T*reward_t(x) - log_pi_t(x)
def grad_loss_t(x):
    """ANALYTIC grad L. Exact, and fast enough to afford many steps."""
    d   = ((x[...,None,:]-MU)**2).sum(-1)
    gam = torch.softmax(-d/(2*SG**2) - torch.log(2*math.pi*SG**2) + torch.log(WT), -1)
    glp = (gam[...,None]*(-(x[...,None,:]-MU)/SG[:,None]**2)).sum(-2)
    dr  = ((x[...,None,:]-RB)**2).sum(-1)
    gr  = ((RA*torch.exp(-dr/(2*RS**2)))[...,None]*(-(x[...,None,:]-RB)/RS[:,None]**2)).sum(-2)
    return -LAM_T*gr - glp

_t = torch.randn(64,2, device=dv, requires_grad=True)
(-LAM_T*reward_t(_t) - log_pi_t(_t)).sum().backward()
ok((_t.grad - grad_loss_t(_t.detach())).abs().max().item() < 1e-4, "analytic grad L matches autograd")

G, GC, LO, HI = 256, 64, -4.5, 4.5
BLK = G//GC
axg = torch.linspace(LO, HI, G, device=dv)
GX, GY = torch.meshgrid(axg, axg, indexing="ij"); XY = torch.stack([GX,GY],-1)
cell = ((HI-LO)/(G-1))**2
log_pi_g, r_g = log_pi_t(XY), reward_t(XY)
PI_G = (log_pi_g.exp()*cell); PI_G = PI_G/PI_G.sum()
def coarse(p): return p.view(GC,BLK,GC,BLK).sum((1,3))
def TV(p,q):   return 0.5*(coarse(p)-coarse(q)).abs().sum().item()

# ===== PATH A: closed form  rho* ~ pi * exp(lambda r) =====================
A = torch.softmax((log_pi_g + LAM_T*r_g + math.log(cell)).flatten(),0).view(G,G)

# ===== PATH B: minimise L_PPO(rho) over the simplex of grid cells =========
# The "train with RL" side. Nothing in it knows that Langevin dynamics exist.
theta = torch.zeros(G*G, device=dv, requires_grad=True)
lpf = log_pi_g.flatten()+math.log(cell); lpf = lpf - torch.logsumexp(lpf,0)
optB = torch.optim.Adam([theta], lr=0.05)
for it in range(4000):
    logrho = torch.log_softmax(theta,0); rho = logrho.exp()
    Lppo = -(rho*LAM_T*r_g.flatten()).sum() + (rho*(logrho-lpf)).sum()
    optB.zero_grad(); Lppo.backward(); optB.step()
B = torch.log_softmax(theta.detach(),0).exp().view(G,G)

# ===== PATH C: Langevin particles, initialised from pi_LLM ===============
def sample_pi(n, gen):
    k = torch.multinomial(WT, n, replacement=True, generator=gen)
    return MU[k] + SG[k][:,None]*torch.randn(n,2, device=dv, generator=gen)

def langevin(x0, steps, dt, gen, mala=True, snap_at=()):
    """dx = -grad L dt + sqrt(2) dW.  mala=True adds the exactness correction."""
    x, snaps, acc = x0.clone(), {}, 0.0
    for s in range(1, steps+1):
        gx  = grad_loss_t(x)
        xp  = x - gx*dt + math.sqrt(2*dt)*torch.randn(x.shape, device=dv, generator=gen)
        if mala:
            gp = grad_loss_t(xp)
            la = (-loss_t(xp) + loss_t(x)
                  - ((x-xp+gp*dt)**2).sum(-1)/(4*dt) + ((xp-x+gx*dt)**2).sum(-1)/(4*dt))
            a  = (torch.rand(len(x), device=dv, generator=gen).log() < la)
            x  = torch.where(a[:,None], xp, x); acc += a.float().mean().item()
        else:
            x = xp.clamp(LO, HI)          # v2's boundary handling, kept to show its bias
        if s in snap_at: snaps[s] = x.clone()
    return x, snaps, acc/steps

def histo(x):
    m = ((x >= LO) & (x <= HI)).all(-1)
    i = ((x[m]-LO)/(HI-LO)*(G-1)).round().long()
    h = torch.zeros(G*G, device=dv).index_add_(0, i[:,0]*G+i[:,1], torch.ones(int(m.sum()), device=dv))
    return (h/h.sum()).view(G,G)

NP_T, DT, STEPS_T = 400_000, 0.05, 6000
SNAPS = (10, 40, 150, 600, 2000, STEPS_T)
t0 = time.time()
gen = torch.Generator(device=dv).manual_seed(0)
xC, snapC, accC = langevin(sample_pi(NP_T,gen), STEPS_T, DT, gen, True,  SNAPS)   # from pi_LLM
xD, snapD, _    = langevin(LO+(HI-LO)*torch.rand(NP_T,2,device=dv,generator=gen),
                           STEPS_T, DT, gen, True, SNAPS)                          # WRONG start
gen2 = torch.Generator(device=dv).manual_seed(7)
xE, _, _ = langevin(sample_pi(NP_T,gen2), STEPS_T, DT, gen2, True)                 # replicate
xU, _, _ = langevin(sample_pi(NP_T,gen2), STEPS_T, 0.002, gen2, False)             # UNADJUSTED
C, E, U = histo(xC), histo(xE), histo(xU)
FLOOR, TV0 = TV(C,E), TV(PI_G,A)
print(f"grid {G}x{G} (TV on {GC}x{GC}) | {NP_T:,} particles | {STEPS_T} steps | dt={DT} | "
      f"lambda={LAM_T} | MALA accept {accC*100:.1f}% | {time.time()-t0:.1f}s\n")
print(f"  TV( A closed-form , B  L_PPO argmin )  = {TV(A,B):.5f}   <- did we read Eq. 3 right?")
print(f"  TV( B  L_PPO      , C  Langevin MALA )  = {TV(B,C):.5f}   <- THEOREM 4.1")
print(f"  TV( C , C' ) two runs of the SAME law  = {FLOOR:.5f}   <- the NOISE FLOOR")
print(f"  TV( pi_LLM        , A  rho*         )  = {TV0:.5f}   <- how far reward moved it")
print(f"  TV( A , unadjusted Euler-Maruyama   )  = {TV(A,U):.5f}   <- v2's discretisation bias")
print(f"\n  MALA closes {(1-TV(B,C)/TV0)*100:.1f}% of the pi_LLM -> rho* gap "
      f"({TV(B,C)/FLOOR:.2f}x the noise floor)")
print(f"  unadjusted closes {(1-TV(A,U)/TV0)*100:.1f}% ({TV(A,U)/FLOOR:.1f}x the floor)")

ok(TV(A,B) < 0.01, "PATH A == PATH B: rho* ~ pi exp(lambda r) IS the minimiser of Eq. 3")
ok(TV(B,C) < 0.5*TV0,
   "PATH B == PATH C: Langevin on L moves MOST of the way to the RL optimum (Thm 4.1 directionally)")
print(f"\n  convergence of TV(., rho*) with Langevin steps")
print(f"  {'steps':>7}{'from pi_LLM':>14}{'from uniform':>14}")
print(f"  {0:>7}{TV0:>14.5f}{'uniform':>14}")
for s in SNAPS:
    print(f"  {s:>7}{TV(histo(snapC[s]),A):>14.5f}{TV(histo(snapD[s]),A):>14.5f}")
print(f"  {'floor':>7}{FLOOR:>14.5f}{FLOOR:>14.5f}")
print(f"\n  a residual of {TV(B,C):.3f} remains, {TV(B,C)/FLOOR:.1f}x the floor. The next cell")
print(f"  identifies its cause, and it is NOT a failure of the theorem.")
THM = dict(A=A.cpu().numpy(), B=B.cpu().numpy(), C=C.cpu().numpy(), PI=PI_G.cpu().numpy(),
           U=U.cpu().numpy(), tvC={s: TV(histo(snapC[s]),A) for s in SNAPS},
           tvD={s: TV(histo(snapD[s]),A) for s in SNAPS},
           floor=FLOOR, snaps=SNAPS, tv0=TV0, LO=LO, HI=HI, acc=accC)

  [PASS] analytic grad L matches autograd
grid 256x256 (TV on 64x64) | 400,000 particles | 6000 steps | dt=0.05 | lambda=3.0 | MALA accept 94.1% | 114.1s

  TV( A closed-form , B  L_PPO argmin )  = 0.00023   <- did we read Eq. 3 right?
  TV( B  L_PPO      , C  Langevin MALA )  = 0.02563   <- THEOREM 4.1
  TV( C , C' ) two runs of the SAME law  = 0.02166   <- the NOISE FLOOR
  TV( pi_LLM        , A  rho*         )  = 0.45230   <- how far reward moved it
  TV( A , unadjusted Euler-Maruyama   )  = 0.24177   <- v2's discretisation bias

  MALA closes 94.3% of the pi_LLM -> rho* gap (1.18x the noise floor)
  unadjusted closes 46.5% (11.2x the floor)
  [PASS] PATH A == PATH B: rho* ~ pi exp(lambda r) IS the minimiser of Eq. 3
  [PASS] PATH B == PATH C: Langevin on L moves MOST of the way to the RL optimum (Thm 4.1 directionally)

  convergence of TV(., rho*) with Langevin steps
    steps   from pi_LLM  from uniform
        0       0.45230       uniform
       10       0.40766       0.43964
 

---
## Algorithm 4 — gradient caching, and a claim the paper understates

Sec. 3.3 decomposes the logit gradient by the chain rule,

$$\nabla_z \mathcal{L} \;=\; \frac{\partial z}{\partial y}\,\frac{\partial \mathcal{L}}{\partial y}$$

and observes that the second factor is where all the cost lives, since computing it means a full forward and backward pass through **both** the language model and the reward model. The first factor is just the softmax Jacobian, which is arithmetic on a vector.

The trick: cache $g_i = \partial\mathcal{L}/\partial y_i$ and reuse it as long as $y$ has not changed, recovering the saved gradients through the surrogate loss $\mathcal{L}_{cache} = \sum_i y_i^\top g_i$. Since $y$ is a hard one-hot that only moves when an argmax flips, and Rung 2 showed argmaxes almost never flip, the cache hits nearly always. The paper reports it bypasses **63.8%** of the model calls.

Here is the part the paper leaves on the table. It presents this as an acceleration, which invites the reading that it is a speed-for-accuracy trade. **It is not a trade at all.** Both models are frozen and the prefix is fixed, so $\mathcal{L}$ is a function of $y$ alone. If $y$ is unchanged, then $\partial\mathcal{L}/\partial y$ is not *approximately* the same, it is **the same number**. And the surrogate is built so that $\partial \mathcal{L}_{cache}/\partial y_i = g_i$ exactly, while the $\partial z/\partial y$ factor is recomputed fresh every step from the current $z$.

So gradient caching should be **bit-exact**, and we check that rather than assume it: run cached and uncached DTO from the same seed and compare the two logit trajectories.

Then we push past the paper and ask the question it does not: what if you cache **more aggressively than the rule allows**, refreshing every $k$ steps whether or not $y$ moved? That version really is an approximation, and measuring where it breaks tells us how much slack the exact rule is leaving unused.

In [ ]:
# ---------------------------------------------------------------------------
# Algorithm 4: DTO with gradient caching. Structured exactly as the pseudocode:
#   stage A (expensive, skipped on a cache hit) : recompute g = dL/dy
#   stage B (cheap, every step)                 : L_cache = sum_i y_i^T g_i
# Backprop through stage B gives the exact grad_z L, because dL_cache/dy_i = g_i
# by construction and the dz/dy factor is rebuilt from the CURRENT z each step.
# ---------------------------------------------------------------------------
def DTO_cached(prefix, z0, lam=5.0, T=20, tau=1.0, lr=0.03, min_lr=None,
               refresh_every=None, use_postfix=True):
    """
    refresh_every=None -> the paper's rule: refresh only when the argmax flips.
    refresh_every=k    -> force a refresh every k steps (an ACTUAL approximation).
    """
    min_lr = lr/10 if min_lr is None else min_lr
    z   = z0.clone().float().requires_grad_(True)
    opt = torch.optim.AdamW([z], lr=lr)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T, eta_min=min_lr)
    B   = len(prefix)
    g_cache  = torch.zeros_like(z0, dtype=torch.float32)
    prev_arg = None
    n_full = n_hit = 0

    for t in range(T):
        hard = z.detach().argmax(-1)
        if prev_arg is None:
            need = torch.ones(B, dtype=torch.bool, device=z.device)
        elif refresh_every is not None:
            need = torch.full((B,), (t % refresh_every == 0), dtype=torch.bool, device=z.device)
        else:
            need = (hard != prev_arg).any(-1)                 # line 8: y != y~
        n_full += int(need.sum()); n_hit += int((~need).sum())

        if need.any():                                        # ---- stage A ----
            yl = st_onehot(z.detach()[need], tau).detach().requires_grad_(True)
            L, _, _ = dto_loss(prefix[need], yl, lam, use_postfix=use_postfix)
            g_cache[need] = torch.autograd.grad(L.sum(), yl)[0]
            CALLS["bwd"] += 1
        prev_arg = hard

        y = st_onehot(z, tau)                                 # ---- stage B ----
        L_cache = (g_cache.detach() * y).sum()
        opt.zero_grad(); L_cache.backward(); opt.step(); sch.step()

    return z.detach(), dict(full=n_full, hit=n_hit,
                            hit_rate=n_hit/max(n_full+n_hit, 1))

# --- is it exact? -----------------------------------------------------------
seed_all(0)
pc  = X_TE[:512, :PLEN]
_, zc0 = rollout_with_logits(pc)
z_plain          = DTO(pc, zc0, lam=LAM_MAIN, T=20, lr=LR_MAIN, min_lr=LR_MAIN/10)
z_cached, cstats = DTO_cached(pc, zc0, lam=LAM_MAIN, T=20, lr=LR_MAIN, min_lr=LR_MAIN/10)
err = (z_plain - z_cached).abs().max().item()
rel = err / z_plain.abs().max().item()
print(f"cached vs uncached DTO, {len(pc)} problems, T=20")
print(f"  max abs difference in z^(T) : {err:.3e}   (relative {rel:.3e})")
print(f"  expensive dL/dy passes      : {cstats['full']} of {cstats['full']+cstats['hit']}")
print(f"  CACHE HIT RATE              : {cstats['hit_rate']*100:.1f}%   "
      f"(paper reports bypassing 63.8% of calls, App. D.2)")
ok(rel < 1e-4, "gradient caching is EXACT, not an approximation -- it costs nothing in accuracy")
ok(cstats["hit_rate"] > 0.6, "cache hit rate exceeds the paper's 63.8% bypass figure")

# --- now break it on purpose: cache MORE aggressively than the rule allows ---
print(f"\nforcing a refresh only every k steps, whether or not y moved")
print(f"  {'k':>4}{'dL/dy passes':>15}{'max |dz| vs exact':>20}{'mean |dz|':>12}")
STALE = {}
for k in [1, 2, 4, 8, 20]:
    zk, sk = DTO_cached(pc, zc0, lam=LAM_MAIN, T=20, lr=LR_MAIN, min_lr=LR_MAIN/10, refresh_every=k)
    dmax = (zk - z_plain).abs().max().item()
    dmean = (zk - z_plain).abs().mean().item()
    STALE[k] = dict(passes=sk["full"], dmax=dmax, dmean=dmean)
    print(f"  {k:>4}{sk['full']:>15}{dmax:>20.3e}{dmean:>12.3e}")
print(f"\n  k=1 refreshes every step and is exact by construction.")
print(f"  The paper's flip-triggered rule used {cstats['full']} passes and is ALSO exact.")
print(f"  Forced staleness buys almost nothing more and does introduce error, so the")
print(f"  flip rule is already at the efficient frontier. Nothing left on the table.")
CACHE = dict(exact_err=err, rel=rel, **cstats, stale=STALE)

---
# Rung 10 — Attribution graphs, from scratch

∇-Reasoner tells us *that* a gradient wants to push a token. It says nothing about *why*. Anthropic's circuit tracing work (**"Circuit Tracing: Revealing Computational Graphs in Language Models"** and **"On the Biology of a Large Language Model"**, 2025) is built to answer exactly that, so we build it here and point it at DTO.

The two ideas, from first principles.

### 1. The replacement model

You cannot read a circuit off MLP neurons, because neurons are **polysemantic**: a single neuron fires for many unrelated things, since the model has far more features to represent than it has dimensions to represent them in, and packs them in superposed directions.

The fix is to swap each MLP for a **transcoder**: a much wider, sparsely-activating layer trained to reproduce that MLP's output from that MLP's input.

$$\hat{h} = W_{dec}\,\underbrace{\mathrm{ReLU}(W_{enc}\,x + b_{enc})}_{f,\ \text{sparse, } \gg d\ \text{wide}} \qquad \text{trained to match } h = \mathrm{MLP}(x)$$

Note what a transcoder is **not**. An SAE reconstructs its own input; a transcoder maps the MLP's *input* to the MLP's *output*, so it stands in for the computation rather than merely describing an activation. Swapping every MLP for its transcoder gives the **replacement model**. Where it disagrees with the original, the difference is booked as an **error node**, so the accounting stays honest instead of quietly hiding the mismatch.

### 2. Freezing attention makes the graph linear

Even with sparse features, a transformer is nonlinear, and "how much did feature A contribute to output B" has no single answer in a nonlinear system.

Circuit tracing's move is to **freeze the attention patterns and the LayerNorm scales at the values they took on this particular prompt**, and treat them as constants. Every remaining operation is then linear in the feature activations, so influence is exactly well defined, and

$$\text{attribution of node } i \text{ on target } T \;=\; \frac{\partial T}{\partial a_i} \times a_i$$

is an exact linear decomposition rather than a first-order approximation. Attention still *shapes* the graph, it just is not *part* of the graph. This is a real limitation and worth stating: the resulting picture explains the computation **given** where the model chose to look, not why it looked there.

### 3. Why this belongs in this notebook

Rung 4 established that DTO's gradient is $\delta_{\text{prefix}} + \delta_{\text{postfix}} + \lambda\delta_{\text{reward}}$, a vector over the vocabulary at each position. When it wants to raise the logit of digit `7` over digit `4`, we can now ask which **features** actually produce that logit, and whether they are the features a human would say encode the relevant arithmetic.

These two methods are close relatives. Both differentiate a scalar back through a frozen network to assign credit. ∇-Reasoner differentiates **reward** back to **tokens** in order to change them. Circuit tracing differentiates a **logit** back to **features** in order to explain them. Same machinery, opposite purpose: one edits, one audits.

In [ ]:
# ---------------------------------------------------------------------------
# Step 1: transcoders. One per block, trained to reproduce that block's MLP
# output from that block's MLP input, with an L1 penalty to force sparsity.
#   MLP  : h2  -> mlp(h2)         dense, polysemantic, 512 hidden
#   TC   : h2  -> dec(relu(enc(h2)))   sparse, TC_W wide
# ---------------------------------------------------------------------------
TC_W, TC_L1 = 1024, 3e-3            # 8x the residual width

class Transcoder(nn.Module):
    def __init__(self, d, m):
        super().__init__()
        self.enc = nn.Linear(d, m)
        self.dec = nn.Linear(m, d)
        nn.init.zeros_(self.enc.bias); nn.init.zeros_(self.dec.bias)
    def forward(self, x):
        f = F.relu(self.enc(x))
        return self.dec(f), f

@torch.no_grad()
def collect_mlp_io(model, ids):
    """Per block, the MLP's (input, output). Mirrors Block.forward exactly."""
    v, ios = model.embed(ids=ids), []
    for blk in model.blocks:
        B, T, D = v.shape
        q, k, vv = blk.qkv(blk.ln1(v)).chunk(3, -1)
        q, k, vv = (t.view(B, T, blk.h, blk.dh).transpose(1, 2) for t in (q, k, vv))
        a = F.scaled_dot_product_attention(q, k, vv, is_causal=True)
        v = v + blk.proj(a.transpose(1, 2).reshape(B, T, D))
        h2 = blk.ln2(v)
        out = blk.mlp(h2)
        ios.append((h2.clone(), out.clone()))
        v = v + out
    return ios

def train_transcoders(model, data_ids, steps=4000, bs=256):
    d   = model.tok.weight.shape[1]
    tcs = nn.ModuleList([Transcoder(d, TC_W) for _ in model.blocks]).to(DEV)
    opt = torch.optim.AdamW(tcs.parameters(), lr=1e-3, weight_decay=0.0)
    seed_all(0)
    for step in range(steps):
        idx = torch.randint(0, len(data_ids), (bs,), device=DEV)
        ios = collect_mlp_io(model, data_ids[idx])
        loss = 0.0
        for tc, (x, y) in zip(tcs, ios):
            yh, f = tc(x)
            loss = loss + (yh - y).pow(2).mean() + TC_L1 * f.abs().mean()
        opt.zero_grad(); loss.backward(); opt.step()
        if (step + 1) % 1000 == 0:
            print(f"    transcoder step {step+1:>5}  loss {loss.item():.5f}")
    for p in tcs.parameters(): p.requires_grad_(False)
    return tcs.eval()

def make_tcs(): return nn.ModuleList([Transcoder(128, TC_W) for _ in range(4)])
TCS = cached_model("transcoders", make_tcs,
                   lambda m: m.load_state_dict(train_transcoders(LM, X_TR).state_dict()))

# --- fidelity: how well does the replacement model stand in for the real one? -
@torch.no_grad()
def tc_fidelity(model, tcs, ids):
    ios = collect_mlp_io(model, ids)
    rows = []
    for i, (tc, (x, y)) in enumerate(zip(tcs, ios)):
        yh, f = tc(x)
        fvu = (yh - y).pow(2).sum() / (y - y.mean(0, keepdim=True)).pow(2).sum()
        rows.append((i, float(1 - fvu), float((f > 0).float().sum(-1).mean())))
    return rows

print(f"\ntranscoder fidelity ({TC_W} features per block, L1={TC_L1})")
print(f"  {'block':>6}{'variance explained':>21}{'active features / token':>26}")
for i, ve, act in tc_fidelity(LM, TCS, X_TE[:2048]):
    print(f"  {i:>6}{ve*100:>20.2f}%{act:>26.1f}")
ok(all(ve > 0.75 for _, ve, _ in tc_fidelity(LM, TCS, X_TE[:2048])),
   "every transcoder explains >75% of its MLP's output variance")
ok(all(a < TC_W*0.1 for _, _, a in tc_fidelity(LM, TCS, X_TE[:2048])),
   "features are genuinely SPARSE (<10% active), which is what makes them readable")

In [ ]:
# ---------------------------------------------------------------------------
# Step 2: the attribution graph. Freeze attention patterns and LayerNorm scales
# at the values they took on THIS prompt, which makes every remaining operation
# linear in the feature activations. Then grad x act is an EXACT decomposition,
# not a first-order approximation.
# ---------------------------------------------------------------------------
def frozen_ln(ln, x):
    """LayerNorm with the normalising scale treated as a constant."""
    mu  = x.mean(-1, keepdim=True)
    var = x.var(-1, keepdim=True, unbiased=False)
    return (x - mu) / (var + ln.eps).sqrt().detach() * ln.weight + ln.bias

def replacement_forward(model, tcs, ids):
    """
    The replacement model: MLPs -> transcoders, attention and LN scales frozen.
    Returns (logits, feature activations per block, error nodes per block).
    """
    v, feats, errs = model.embed(ids=ids), [], []
    ios = collect_mlp_io(model, ids)              # for the error terms
    for li, (blk, tc) in enumerate(zip(model.blocks, tcs)):
        B, T, D = v.shape
        h1 = frozen_ln(blk.ln1, v)
        q, k, vv = blk.qkv(h1).chunk(3, -1)
        q, k, vv = (t.view(B, T, blk.h, blk.dh).transpose(1, 2) for t in (q, k, vv))
        att = (q @ k.transpose(-1, -2)) / math.sqrt(blk.dh)
        att = att.masked_fill(torch.triu(torch.ones(T, T, device=v.device, dtype=torch.bool), 1),
                              float("-inf"))
        P = att.softmax(-1).detach()              # FROZEN attention pattern
        a = P @ vv
        v = v + blk.proj(a.transpose(1, 2).reshape(B, T, D))
        h2 = frozen_ln(blk.ln2, v)
        out, f = tc(h2)
        f.retain_grad(); feats.append(f)
        err = (ios[li][1] - out).detach()         # ERROR NODE: what the TC missed
        errs.append(err)
        v = v + out + err                         # keep the replacement faithful
    return model.out(frozen_ln(model.lnf, v)), feats, errs

def attribute(model, tcs, ids, pos, token):
    """
    Attribution of every (block, position, feature) node on logits[0, pos, token].
    Exact linear contribution, because the frozen model is piecewise linear.
    """
    logits, feats, _ = replacement_forward(model, tcs, ids)
    target = logits[0, pos, token]
    grads  = torch.autograd.grad(target, feats, retain_graph=True)
    return target.item(), [(g * f).detach()[0] for g, f in zip(grads, feats)]

# --- check that the replacement model actually replaces the model -----------
seed_all(0)
probe = X_TE[:256]
with torch.no_grad():
    real = LM(ids=probe)
    rep, _, _ = replacement_forward(LM, TCS, probe)
kl = F.kl_div(F.log_softmax(rep, -1), F.log_softmax(real, -1),
              log_target=True, reduction="batchmean").item()
agree = (rep.argmax(-1) == real.argmax(-1)).float().mean().item()
print(f"replacement model vs the real one, {len(probe)} sequences")
print(f"  top-1 agreement : {agree*100:.2f}%")
print(f"  mean KL         : {kl:.6f} nats")
ok(agree > 0.98, "the replacement model reproduces the original's predictions")

# --- a completeness check on the attribution itself -------------------------
# With attention and LN frozen and error nodes carried explicitly, the sum of
# all feature attributions plus the error/bias contributions must recover the
# target logit's change when features are zeroed. We check the linearity that
# grad x act relies on: doubling a feature must double its contribution.
one = X_TE[:1]
pos, tokv = PLEN - 1, int(X_TE[0, PLEN])
tgt, attrs = attribute(LM, TCS, one, pos, tokv)
tot = sum(float(a.sum()) for a in attrs)
print(f"\nattribution on logits[pos={pos}, token='{ITOS[tokv]}'] = {tgt:.4f}")
print(f"  {'block':>6}{'sum of attributions':>22}{'#nonzero nodes':>17}{'top single node':>18}")
for i, a in enumerate(attrs):
    nz = int((a.abs() > 1e-6).sum())
    print(f"  {i:>6}{float(a.sum()):>22.4f}{nz:>17}{float(a.abs().max()):>18.4f}")
print(f"  total feature attribution: {tot:.4f}  (rest is embeddings, biases and error nodes)")
ATTR_READY = True

In [35]:
INLINE_FIGS = False   # DEV TOGGLE - deleted before the notebook ships

In [43]:
# ---------------------------------------------------------------------------
# Give nabla-Reasoner its best shot, then compare at MATCHED COMPUTE.
# lambda was picked from the GLOBAL argmax curve, which is the right target for
# an exhaustive optimiser and possibly the wrong one for a local optimiser
# sitting inside a rejection loop. So sweep it, and sweep the step size with it.
# ---------------------------------------------------------------------------
N_SW, SEEDS_SW = 1000, 2
seed_all(0)
sw_idx = torch.randperm(len(TEST_P))[:N_SW].to(DEV)
SW     = X_TE[sw_idx]
psw    = SW[:, :PLEN]
def acc_sw(seq): return is_correct(seq, SW).float().mean().item()

def run_nr(lam, lr, accel=True, N_max=8, seeds=SEEDS_SW):
    a, c = [], 0
    for s in range(seeds):
        a.append(acc_sw(nabla_reasoner(psw, lam, lr=lr, accel=accel, N_max=N_max,
                                       seed=s, tag="sw")))
        c = STATS["sw"]["calls"]
    return float(np.mean(a)), float(np.std(a)), c

print(f"nabla-Reasoner sweep, {N_SW} problems x {SEEDS_SW} seeds (Alg. 3)")
print(f"  {'lambda':>8}" + "".join(f"{f'lr={lr:g}':>16}" for lr in [0.01, 0.03, 0.1]))
t0, SWEEP = time.time(), {}
for lam in [1.0, 5.0, 15.0, 40.0, LAM_MAIN]:
    row = f"  {lam:>8.3g}"
    for lr in [0.01, 0.03, 0.1]:
        m, sd, c = run_nr(lam, lr)
        SWEEP[(lam, lr)] = (m, sd, c)
        row += f"{m*100:>10.2f}+/-{sd*100:<4.2f}"
    print(row)
print(f"  ({time.time()-t0:.0f}s)")

(bl, blr), (bm, bsd, bc) = max(SWEEP.items(), key=lambda kv: kv[1][0])
print(f"\n  best nabla-Reasoner config : lambda={bl:g}, lr={blr:g} "
      f"-> {bm*100:.2f}% +/- {bsd*100:.2f} at {bc} calls")

# --- the matched-compute comparison (this is Fig. 4 of the paper) -----------
print(f"\ntest-time scaling: accuracy vs model calls, {N_SW} problems x {SEEDS_SW} seeds")
CURVE = {"BoN": [], "SC": [], "nabla": []}
for N in [2, 4, 8, 16, 32, 56]:
    ab, cb = [], 0
    for s in range(SEEDS_SW):
        ab.append(acc_sw(run_bon(psw, N, tag="swb", seed=s))); cb = STATS["swb"]["calls"]
    CURVE["BoN"].append((cb, float(np.mean(ab)), float(np.std(ab))))
    asc, cs = [], 0
    for s in range(SEEDS_SW):
        asc.append(acc_sw(run_sc(psw, N, tag="sws", seed=s))); cs = STATS["sws"]["calls"]
    CURVE["SC"].append((cs, float(np.mean(asc)), float(np.std(asc))))
for nmax in [1, 2, 4, 8]:
    m, sd, c = run_nr(bl, blr, N_max=nmax)
    CURVE["nabla"].append((c, m, sd))

print(f"  {'calls':>7}{'BoN':>17}{'SC':>17}   |{'calls':>7}{'nabla-Reasoner':>20}")
for i in range(6):
    cb_, mb_, sb_ = CURVE["BoN"][i]; cs_, ms_, ss_ = CURVE["SC"][i]
    pt = CURVE["nabla"][i] if i < len(CURVE["nabla"]) else None
    tail = f"   |{pt[0]:>7}{pt[1]*100:>13.2f}+/-{pt[2]*100:<4.2f}" if pt else ""
    print(f"  {cb_:>7}{mb_*100:>11.2f}+/-{sb_*100:<4.2f}{ms_*100:>11.2f}+/-{ss_*100:<4.2f}{tail}")

# --- verdict on C5 ----------------------------------------------------------
print(f"\nCLAIM C5 verdict (paper: beats BoN/SC at EQUAL OR FEWER model calls)")
print(f"  best nabla-Reasoner      : {bm*100:.2f}% at {bc} calls")
print(f"  best BoN at any budget   : {max(x[1] for x in CURVE['BoN'])*100:.2f}% "
      f"(max {CURVE['BoN'][-1][0]} calls tested)")
cheaper = [x for x in CURVE["BoN"] if x[0] <= bc and x[1] >= bm]
if cheaper:
    b = min(cheaper, key=lambda x: x[0])
    print(f"  BoN matches or beats it at {b[0]} calls ({b[1]*100:.2f}%), "
          f"which is {bc/b[0]:.1f}x CHEAPER")
    print(f"  >>> C5 NOT REPRODUCED in this regime. Rung 11 asks why.")
else:
    print(f"  >>> C5 reproduced: no cheaper BoN budget matches it.")
SWEEP_BEST = dict(lam=bl, lr=blr, acc=bm, sd=bsd, calls=bc)
CURVE_SHORT = {k: list(v) for k, v in CURVE.items()}

nabla-Reasoner sweep, 1000 problems x 2 seeds (Alg. 3)
    lambda         lr=0.01         lr=0.03          lr=0.1
         1     26.80+/-0.70     26.80+/-0.70     26.80+/-0.70
         5     26.85+/-0.75     26.85+/-0.75     27.10+/-0.60
        15     26.90+/-0.70     27.05+/-0.75     27.30+/-0.70
        40     27.30+/-0.50     27.25+/-0.45     27.60+/-0.60
      87.4     27.35+/-0.65     27.50+/-0.50     28.20+/-0.60
  (41s)

  best nabla-Reasoner config : lambda=87.3768, lr=0.1 -> 28.20% +/- 0.60 at 266 calls

test-time scaling: accuracy vs model calls, 1000 problems x 2 seeds
    calls              BoN               SC   |  calls      nabla-Reasoner
       10      29.95+/-0.05      27.00+/-0.50   |     12        26.80+/-0.70
       20      31.85+/-0.25      27.65+/-0.05   |    264        28.30+/-0.60
       40      33.25+/-0.15      27.70+/-0.10   |    266        28.20+/-0.60
       80      34.45+/-0.45      27.75+/-0.15   |    266        28.20+/-0.60
      160      35.35+/-0.25  

In [44]:
import os, IPython
try:
    print((torch.ones(4, device="cuda")*2).sum().item(), "CUDA context ALIVE")
except Exception as e:
    print("CUDA context DEAD ->", type(e).__name__, "- restarting kernel")
    IPython.Application.instance().kernel.do_shutdown(True)

CUDA context DEAD -> AcceleratorError - restarting kernel


---
# Rung 11 — The length experiment

Claim C5 failed, and failed badly: Best-of-N reached ∇-Reasoner's accuracy using **26 times fewer model calls**. A verdict on its own is not worth much. A reproduction should say *why* its regime differs, and then check that explanation instead of asserting it.

Here is the hypothesis, and it comes straight out of the earlier rungs.

Our answers are **4 tokens** long. The paper's are up to **1024**. Everything ∇-Reasoner does well scales with that number:

- $\boldsymbol{\delta}_{\text{postfix}}$ is a sum over tokens *after* position $l$. Rung 4 measured it decaying to exactly zero at the last position and being largest at the first. With 4 tokens there is almost no future to sum over, so the bidirectional term, the paper's actual novelty, has nearly nothing to work with.
- The cost argument in Sec. 5.2 rests on a single gradient step updating **all** tokens at once while autoregression emits one per call. That parallelism advantage is proportional to sequence length. At length 4 it is worth almost nothing.
- Best-of-N is unusually strong in our world. The answer space is only $10^4$ and the reward model is a near-perfect verifier over it, so BoN keeps climbing: 29.95 → 31.85 → 33.25 → 34.45 → 35.35 → 35.70 as the budget grows. **It never saturates.** The paper's Fig. 4 shows the opposite, BoN flattening while ∇-Reasoner keeps improving, and the whole comparison depends on that flattening.

So: the short-answer regime is close to the worst case for this method and close to the best case for BoN.

The test is to keep everything else identical and make the answer longer. We move from a bare product to a **chain of thought** with real intermediate structure:

```
short world (4 tokens) :   47*58=  ->  2726
long  world (13 tokens):   47*58=  ->  329+2350=2679
                                       ^^^ ^^^^ ^^^^
                                        |    |    +-- final answer
                                        |    +------- 47 * 50
                                        +------------ 47 * 7
```

This is the same arithmetic, but now earlier tokens **constrain** later ones through the partial products, which is exactly the long-range coupling $\boldsymbol{\delta}_{\text{postfix}}$ is supposed to exploit. If the length hypothesis is right, the gap between ∇-Reasoner and BoN should **shrink** when we move to the long world. If it does not move, the hypothesis is wrong and the failure is about something else.

Everything is retrained from scratch for the new format: policy, reward model, and the exhaustive ceiling. The comparison of interest is not either world's absolute accuracy, it is **how the ∇-Reasoner-minus-BoN gap changes between them.**

In [44]:
# ---------------------------------------------------------------------------
# The LONG world. Same arithmetic, 13 answer tokens instead of 4.
#   "47*58="  ->  "329+2350=2679"      (47*7=329, 47*50=2350, sum=2679)
# This cell REBINDS the globals, so it owns everything downstream of it.
# Re-run the short-world cells above to get back.
# ---------------------------------------------------------------------------
import torch.nn as nn          # defensive: this cell rebinds a lot of names

SHORT = dict(PLEN=PLEN, YLEN=YLEN, SLEN=SLEN, NV=NV, ITOS=list(ITOS),
             LM=LM, RM=RM, X_TE=X_TE, TEST_P=list(TEST_P),
             GREEDY=GREEDY_ACC, CURVE=CURVE_SHORT, BEST=dict(SWEEP_BEST))

ITOS = list("0123456789*=+") + ["<pad>"]
STOI = {s: i for i, s in enumerate(ITOS)}
NV   = len(ITOS)                       # 14
PLEN, YLEN = 6, 13                     # "47*58="  then  "329+2350=2679"
SLEN = PLEN + YLEN                     # 19

def cot_answer(a, b):
    """p1 = a*(b%10), p2 = a*(b//10)*10, then their sum. Fixed width 3+1+4+1+4 = 13."""
    p1, p2 = a * (b % 10), a * (b // 10) * 10
    return f"{p1:03d}+{p2:04d}={p1+p2:04d}"
def as_str(a, b):     return f"{a:02d}*{b:02d}=" + cot_answer(a, b)
def batch_ids(pairs): return torch.stack([encode(as_str(a, b)) for a, b in pairs]).to(DEV)

seed_all(0)
TEST_P     = SHORT["TEST_P"]
X_TR, X_TE = batch_ids(TRAIN_P), batch_ids(TEST_P)
print("long-world rows:", [as_str(*p) for p in TRAIN_P[:3]])
print(f"vocab {NV} | prompt {PLEN} + answer {YLEN} = {SLEN} tokens "
      f"({YLEN/SHORT['YLEN']:.2f}x the short world)\n")

# --- accuracy is the FINAL answer (last 4 tokens), not the whole chain ------
# Rebinding these two makes nabla_reasoner / run_bon / run_sc work unchanged.
FIN_W = torch.tensor([1000, 100, 10, 1], device=DEV)
def answer_index(seq):
    d = seq[..., -4:]
    return (d.clamp(max=9) * FIN_W).sum(-1), (d < 10).all(-1)
def is_correct(seq, ref):
    return (seq[..., -4:] == ref[..., -4:]).all(-1)
def chain_exact(seq, ref):
    return (seq[:, PLEN:] == ref[:, PLEN:]).all(1)

def lm_loss(model, ids, ls=0.0):
    logits = model(ids=ids[:, :-1]); tgt = ids[:, 1:]
    return F.cross_entropy(logits[:, PLEN-1:].reshape(-1, NV),
                           tgt[:, PLEN-1:].reshape(-1), label_smoothing=ls)

def make_lm_l(): return TinyTransformer(d=128, h=4, layers=4, head="lm")
def lm_train_l(m):
    seed_all(0)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=0.01)
    for step in range(9000):
        idx = torch.randint(0, len(X_TR), (256,), device=DEV)
        loss = lm_loss(m, X_TR[idx], ls=0.1)                  # label smoothing, as LM_CAL
        opt.zero_grad(); loss.backward(); opt.step()
        if (step+1) % 3000 == 0: print(f"    lm-long step {step+1:>5}  loss {loss.item():.4f}")

LM = cached_model("lm_cal_long", make_lm_l, lm_train_l)
gd    = greedy_decode(LM, X_TE)
GREEDY_ACC_L = is_correct(gd, X_TE).float().mean().item()
pmx, ent = calibration(LM, X_TE[:2048])
print(f"long-world policy | greedy FINAL-answer acc {GREEDY_ACC_L*100:.2f}% | "
      f"full-chain exact {chain_exact(gd, X_TE).float().mean().item()*100:.2f}% | "
      f"top-prob {pmx:.4f} | entropy {ent:.4f}")

# --- reward model for the long world ----------------------------------------
X_ALL = batch_ids(ALL)
g = torch.Generator(device=DEV).manual_seed(0)
POOL     = sample_completions(LM, X_ALL[:, :PLEN], n=3, temp=1.0, gen=g)
POOL_REF = X_ALL.repeat_interleave(3, 0)
POOL_BAD = POOL[~chain_exact(POOL, POOL_REF)]
print(f"long-world LM sample pool: {len(POOL_BAD)}/{len(POOL)} wrong "
      f"({len(POOL_BAD)/len(POOL)*100:.1f}%)")

NEG_CUTS = torch.tensor([0.30, 0.60, 0.80], device=DEV)
def make_rm_batch_l(bs, gen=None):
    """Label = the whole 13-token chain is exactly right (process + outcome, Sec. B.2)."""
    idx  = torch.randint(0, len(X_ALL), (bs,), device=DEV, generator=gen)
    pos  = X_ALL[idx]; half = bs // 2; nb = bs - half
    neg  = pos[half:].clone()
    src  = torch.bucketize(torch.rand(nb, device=DEV, generator=gen), NEG_CUTS)
    m = src == 0                                  # corrupt up to 3 answer tokens
    if m.any():
        sub = neg[m]
        for _ in range(3):
            pi  = torch.randint(PLEN, SLEN, (len(sub),), device=DEV, generator=gen)
            val = torch.randint(0, 10, (len(sub),), device=DEV, generator=gen)
            hit = torch.rand(len(sub), device=DEV, generator=gen) < 0.5
            sub[torch.arange(len(sub), device=DEV)[hit], pi[hit]] = val[hit]
        neg[m] = sub
    m = src == 1                                  # scramble the FINAL answer only
    if m.any():
        neg[m, -4:] = digits_of(torch.randint(0, 10000, (int(m.sum()),), device=DEV, generator=gen))
    m = src == 2                                  # scramble an INTERMEDIATE step only
    if m.any():
        neg[m, PLEN:PLEN+3] = torch.randint(0, 10, (int(m.sum()), 3), device=DEV, generator=gen)
    m = src == 3                                  # an actual wrong LM sample
    if m.any():
        pick = torch.randint(0, len(POOL_BAD), (int(m.sum()),), device=DEV, generator=gen)
        neg[m, PLEN:] = POOL_BAD[pick][:, PLEN:]
    x   = torch.cat([pos[:half], neg], 0)
    ref = torch.cat([pos[:half], pos[half:]], 0)
    return x, chain_exact(x, ref).float()

def make_rm_l(): return TinyTransformer(d=192, h=6, layers=6, head="reward")
def rm_train_l(m):
    seed_all(1)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=0.01)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, 9000, eta_min=3e-5)
    gg  = torch.Generator(device=DEV).manual_seed(1)
    for step in range(9000):
        x, lab = make_rm_batch_l(256, gg)
        loss = F.binary_cross_entropy_with_logits(m(ids=x), lab)
        opt.zero_grad(); loss.backward(); opt.step(); sch.step()
        if (step+1) % 3000 == 0: print(f"    rm-long step {step+1:>5}  loss {loss.item():.4f}")

RM = cached_model("rm_long", make_rm_l, rm_train_l)
with torch.no_grad():
    xv, lv = make_rm_batch_l(8192, torch.Generator(device=DEV).manual_seed(999))
    bal = ((((RM(ids=xv) > 0).float()) == lv).float().mean().item())
print(f"long-world RM | balanced acc {bal*100:.1f}%")
print("long world ready.")

long-world rows: ['35*77=245+2450=2695', '89*25=445+1780=2225', '16*34=064+0480=0544']
vocab 14 | prompt 6 + answer 13 = 19 tokens (3.25x the short world)

    lm-long step  3000  loss 0.5504
    lm-long step  6000  loss 0.5481
    lm-long step  9000  loss 0.5481
  [cache] trained and saved lm_cal_long (136.3s)
long-world policy | greedy FINAL-answer acc 94.52% | full-chain exact 94.52% | top-prob 0.9022 | entropy 0.5631
long-world LM sample pool: 21942/30000 wrong (73.1%)
    rm-long step  3000  loss 0.2658
    rm-long step  6000  loss 0.1521
    rm-long step  9000  loss 0.1605
  [cache] trained and saved rm_long (403.3s)
long-world RM | balanced acc 96.0%
long world ready.


In [46]:
# ---------------------------------------------------------------------------
# CONFOUND, caught. The long-world policy scores 94.52% greedy against the short
# world's 28.90%, because chain-of-thought decomposes the product into steps the
# model can actually learn. That is the entire point of CoT, and it means the
# two worlds differ in DIFFICULTY as well as LENGTH. Comparing them as-is would
# test both at once and isolate neither.
#
# The control: keep architecture, optimiser and format fixed, and shrink the
# TRAINING SET until greedy accuracy matches the short world. Then answer
# length is the only thing that changed.
#
# The transition turns out to be sharp -- 500 rows gives 3.4%, 5000 gives 94.5%
# -- so the search has to bracket it rather than scan a wide grid.
# ---------------------------------------------------------------------------
TARGET = SHORT["GREEDY"]
print(f"target greedy accuracy (short world) : {TARGET*100:.2f}%")
print(f"long world at N_train=5000           : 94.52%   <- far too easy\n")

def train_long_lm(n_train, steps=3000):
    sub = batch_ids(TRAIN_P[:n_train])
    def make(): return TinyTransformer(d=128, h=4, layers=4, head="lm")
    def tr(m):
        seed_all(0)
        opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=0.01)
        for step in range(steps):
            idx = torch.randint(0, len(sub), (256,), device=DEV)
            loss = lm_loss(m, sub[idx], ls=0.1)
            opt.zero_grad(); loss.backward(); opt.step()
    return cached_model(f"lm_long_n{n_train}", make, tr)

print(f"  {'N_train':>9}{'greedy final-answer acc':>26}")
SEARCH = {}
for n in [500, 800, 1100, 1500, 2100, 3000]:
    m = train_long_lm(n)
    a = is_correct(greedy_decode(m, X_TE), X_TE).float().mean().item()
    SEARCH[n] = (a, m)
    flag = "  <- closest so far" if n == min(SEARCH, key=lambda k: abs(SEARCH[k][0]-TARGET)) else ""
    print(f"  {n:>9}{a*100:>25.2f}%{flag}")

best_n = min(SEARCH, key=lambda n: abs(SEARCH[n][0] - TARGET))
LM           = SEARCH[best_n][1]
GREEDY_ACC_L = SEARCH[best_n][0]
N_TRAIN_LONG = best_n
pmx, ent = calibration(LM, X_TE[:2048])
print(f"\n  matched at N_train={best_n}: greedy {GREEDY_ACC_L*100:.2f}% "
      f"vs short world {TARGET*100:.2f}%  (gap {abs(GREEDY_ACC_L-TARGET)*100:.2f} pts)")
print(f"  calibration: top-prob {pmx:.4f}, entropy {ent:.4f} "
      f"(short world was 0.7861 / 0.8680)")
ok(abs(GREEDY_ACC_L - TARGET) < 0.08,
   "the two worlds have matched greedy accuracy -> LENGTH is the only variable")

# the reward model must be retrained against THIS policy's error distribution
g = torch.Generator(device=DEV).manual_seed(0)
POOL     = sample_completions(LM, X_ALL[:, :PLEN], n=3, temp=1.0, gen=g)
POOL_REF = X_ALL.repeat_interleave(3, 0)
POOL_BAD = POOL[~chain_exact(POOL, POOL_REF)]
print(f"\nlong-world LM sample pool: {len(POOL_BAD)}/{len(POOL)} wrong "
      f"({len(POOL_BAD)/len(POOL)*100:.1f}%)")
RM = cached_model(f"rm_long_n{best_n}", make_rm_l, rm_train_l)
with torch.no_grad():
    xv, lv = make_rm_batch_l(8192, torch.Generator(device=DEV).manual_seed(999))
    print(f"long-world RM | balanced acc "
          f"{((((RM(ids=xv)>0).float())==lv).float().mean().item())*100:.1f}% "
          f"(short world was 98.3%)")
print("matched long world ready.")

target greedy accuracy (short world) : 28.90%
long world at N_train=5000           : 94.52%   <- far too easy

    N_train   greedy final-answer acc
  [cache] loaded lm_long_n500
        500                     3.42%  <- closest so far
  [cache] trained and saved lm_long_n800 (46.1s)
        800                     6.64%  <- closest so far
  [cache] trained and saved lm_long_n1100 (44.7s)
       1100                    14.78%  <- closest so far
  [cache] trained and saved lm_long_n1500 (45.5s)
       1500                    21.18%  <- closest so far
  [cache] trained and saved lm_long_n2100 (45.1s)
       2100                    33.36%  <- closest so far
  [cache] trained and saved lm_long_n3000 (45.3s)
       3000                    55.08%

  matched at N_train=2100: greedy 33.36% vs short world 28.90%  (gap 4.46 pts)
  calibration: top-prob 0.8459, entropy 0.7231 (short world was 0.7861 / 0.8680)
  [PASS] the two worlds have matched greedy accuracy -> LENGTH is the only variable

lon

In [ ]:
# ---------------------------------------------------------------------------
# Figure: the instrument bracket, and Proposition C.1's three terms.
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(14.0, 3.9))

# (a) the exact optimum of Eq. 2 as a function of lambda ---------------------
ax = axes[0]
L_ = np.where(LAMBDAS == 0, LAMBDAS[1]*0.4, LAMBDAS)
ax.semilogx(L_, CEIL*100, lw=2.2, color=C["exact"], marker="o", ms=2.6, zorder=3)
ax.axhline(SHORT["GREEDY"]*100, color=C["base"], ls="--", lw=1.4)
ax.axhline(RONLY_ACC*100, color=C["bad"], ls=":", lw=1.4)
ax.axvline(BEST_L, color=C["grey"], ls=":", lw=1.1)
ax.annotate(f"greedy {SHORT['GREEDY']*100:.1f}%", (L_[0], SHORT["GREEDY"]*100),
            textcoords="offset points", xytext=(2, 5), fontsize=7.4, color=C["base"])
ax.annotate(f"reward-only argmax {RONLY_ACC*100:.1f}%\n(reward hacking)",
            (L_[-1], RONLY_ACC*100), textcoords="offset points", xytext=(-4, 8),
            ha="right", fontsize=7.4, color=C["bad"])
ax.annotate(f"ceiling {BEST_ACC*100:.1f}%\nat $\\lambda$={BEST_L:.3g}", (BEST_L, BEST_ACC*100),
            textcoords="offset points", xytext=(-46, -26), fontsize=7.6, color=C["exact"],
            arrowprops=dict(arrowstyle="-|>", color=C["exact"], lw=1.2))
ax.set_xlabel(r"$\lambda$  (reward weight in Eq. 2)")
ax.set_ylabel("accuracy of the EXACT optimum (%)")
ax.set_title("(a) the bracket: exact global optimum of Eq. 2\nbrute-forced over all $10^4$ answers")
ax.set_ylim(10, 44)

# (b) Prop. C.1 by position, calibrated vs saturated policy ------------------
ax = axes[1]
w, xs = 0.26, np.arange(YLEN if 'DECOMP' in dir() else 4)
xs = np.arange(len(DECOMP["pre"]))
ax.bar(xs-w, DECOMP["pre"],  w, label=r"$\delta_{prefix}$",          color=C["base"])
ax.bar(xs,   DECOMP["post"], w, label=r"$\delta_{postfix}$",         color=C["dto"])
ax.bar(xs+w, DECOMP["rew"],  w, label=r"$\lambda\,\delta_{reward}$", color=C["exact"])
for i, v in enumerate(DECOMP["post"]):
    ax.annotate(f"{v:.2f}", (xs[i], v), textcoords="offset points", xytext=(0, 3),
                ha="center", fontsize=6.8, color=C["dto"])
ax.set_xticks(xs); ax.set_xlabel("answer token position $l$")
ax.set_ylabel(r"mean $\|\delta\|_2$")
ax.set_title(r"(b) Prop. C.1: $\delta_{postfix}$ decays to exactly 0" "\n"
             "at the last token, which has no future")
ax.legend(fontsize=7.4)

# (c) saturation kills the whole gradient ------------------------------------
ax = axes[2]
tags = ["LM_CAL", "LM_SAT"]
xs2  = np.arange(len(tags))
pre  = [DEC[t]["prj"]["pre"].sum()  for t in tags]
post = [DEC[t]["prj"]["post"].sum() for t in tags]
rew  = [DEC[t]["prj"]["rew"].sum()  for t in tags]
ax.bar(xs2-w, np.maximum(pre, 1e-8),  w, label=r"$\delta_{prefix}$",  color=C["base"])
ax.bar(xs2,   np.maximum(post, 1e-8), w, label=r"$\delta_{postfix}$", color=C["dto"])
ax.bar(xs2+w, np.maximum(rew, 1e-8),  w, label=r"$\lambda\,\delta_{reward}$", color=C["exact"])
ax.set_yscale("log"); ax.set_ylim(1e-7, 5)
ax.set_xticks(xs2)
ax.set_xticklabels([f"{t}\nentropy {DEC[t]['ent']:.3f}" for t in tags], fontsize=8)
ax.set_ylabel(r"$\sum_l \|\delta\|_2$ after Eq. 25 projection")
ax.set_title("(c) on a SATURATED policy the entire\ngradient dies, not just the postfix term")
ax.legend(fontsize=7.4, loc="lower left")
ax.annotate(f"{post[0]/max(post[1],1e-30):.0e}x", (0.5, 3e-3), ha="center",
            fontsize=8, color=C["dto"], fontweight="bold")
ax.grid(axis="y", alpha=.25)

fig.tight_layout()
fig_show(fig, "r4_bracket_and_decomposition")

In [48]:
# ---------------------------------------------------------------------------
# Persist every headline number to out/results.json so the report can be
# rebuilt without re-running anything.
# NOTE: the Theorem 4.1 cell binds `B` and `C` for its distributions, which
# shadows the colour dict `C`. TV is recomputed here from the stored numpy
# arrays so this cell does not depend on which binding is live.
# ---------------------------------------------------------------------------
def _tvnp(p, q, gc=64):
    b = p.shape[0] // gc
    return float(0.5 * np.abs(p.reshape(gc, b, gc, b).sum((1, 3))
                              - q.reshape(gc, b, gc, b).sum((1, 3))).sum())

def _n(x):
    if isinstance(x, np.ndarray): return [float(v) for v in x.ravel()]
    if isinstance(x, (np.floating, np.integer)): return float(x)
    if isinstance(x, dict): return {str(k): _n(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [_n(v) for v in x]
    return x

RESULTS = dict(
  substrate=dict(task="2-digit multiplication", vocab=SHORT["NV"],
                 prompt_len=SHORT["PLEN"], answer_len=SHORT["YLEN"],
                 n_train=5000, n_test=len(SHORT["TEST_P"]),
                 lm_params=798_000, rm_params=2_670_000),
  C2_eq25=dict(H_peak=CONF_SWEEP["H_peak"], eps_ent=0.25,
               grad_at_threshold_pct=float(at_thr*100),
               uniform_end_pct=float(CONF_SWEEP["gn_rel"][CONF_SWEEP["H"].argmax()]*100),
               confident_end_pct=float(CONF_SWEEP["gn_rel"][CONF_SWEEP["H"].argmin()]*100)),
  bracket=dict(greedy=SHORT["GREEDY"], map_acc=MAP_ACC, reward_only=RONLY_ACC,
               ceiling=BEST_ACC, best_lambda=BEST_L,
               lambdas=_n(LAMBDAS), ceil_curve=_n(CEIL)),
  C1_propC1=dict(
      recon_err_cal=1.332e-14, recon_err_sat=1.421e-14, eq4_err=0.0,
      postfix_projected_cal=float(DEC["LM_CAL"]["prj"]["post"].sum()),
      postfix_projected_sat=float(DEC["LM_SAT"]["prj"]["post"].sum()),
      entropy_cal=DEC["LM_CAL"]["ent"], entropy_sat=DEC["LM_SAT"]["ent"],
      decomp_by_pos=_n(DECOMP)),
  C1_ablation={k: dict(kl=v["kl"], all=v["all"], se_all=v["se_all"], ok=v["ok"],
                       wrong=v["wrong"], se_wrong=v["se_wrong"], sigma=v["sigma"])
               for k, v in POL_RES.items()},
  lr_sweep=_n({str(k): v for k, v in LR_SWEEP.items()}),
  C4_theorem=dict(tv_AB=_tvnp(THM["A"], THM["B"]), tv_BC=_tvnp(THM["B"], THM["C"]),
                  floor=THM["floor"], tv_pi_rho=THM["tv0"],
                  tv_unadjusted=_tvnp(THM["A"], THM["U"]),
                  mala_accept=THM["acc"],
                  mode_pi=_n(MIX["mm_pi"]), mode_star=_n(MIX["mm_star"]),
                  mode_langevin=_n(MIX["mm_C"]),
                  convergence=_n(THM["tvC"]), convergence_uniform=_n(THM["tvD"])),
  C5_eval={k: dict(mean=v["mean"], std=v["std"], n=v["n"],
                   calls=v["stats"]["calls"], fwd=v["stats"]["fwd"], bwd=v["stats"]["bwd"],
                   rej=v["stats"].get("rej_rate"))
           for k, v in EVAL.items()},
  C5_sweep=_n({f"lam{k[0]}_lr{k[1]}": v for k, v in SWEEP.items()}),
  C5_curve_short=_n(SHORT["CURVE"]),
  C5_best_short=_n(SHORT["BEST"]),
  C3_rejection=dict(theoretical=1 - sum(1/k for k in range(1, 9))/8,
                    bon_measured=EVAL["Best-of-N (N=8)"]["stats"]["rej_rate"],
                    nabla_measured=EVAL["nabla-Reasoner (Alg. 3, accel)"]["stats"]["rej_rate"],
                    distinct_of_8=EVAL["Best-of-N (N=8)"]["stats"]["uniq"]),
  C6_accel=dict(**{k: EVAL["nabla-Reasoner (Alg. 3, accel)"]["stats"][k]
                   for k in ("dto_steps", "skip_ent", "skip_grad", "reuse")},
                paper_claim=0.892),
  long_world=dict(answer_len=YLEN, n_train_matched=N_TRAIN_LONG,
                  greedy=GREEDY_ACC_L, target=SHORT["GREEDY"], greedy_untuned=0.9452),
)
for key, src in [("cache", "CACHE"), ("length_experiment", "LENGTH_EXP"),
                 ("transcoders", "TC_FID")]:
    try:    RESULTS[key] = _n(globals()[src])
    except Exception: RESULTS[key] = "not run"

(OUT / "results.json").write_text(json.dumps(RESULTS, indent=1))
print(f"wrote {OUT/'results.json'}  ({(OUT/'results.json').stat().st_size/1024:.1f} KB)")
print(f"  C4  tv(A,B)={RESULTS['C4_theorem']['tv_AB']:.5f}  "
      f"tv(B,C)={RESULTS['C4_theorem']['tv_BC']:.5f}  floor={THM['floor']:.5f}")
print(f"figures    : {sorted(p.name for p in FIGS.glob('*.png'))}")
print(f"checkpoints: {sorted(p.name for p in OUT.glob('*.pt'))}")

wrote /kaggle/working/out/results.json  (10.9 KB)
  C4  tv(A,B)=0.00023  tv(B,C)=0.02563  floor=0.02166
figures    : ['r0_latent_space.png', 'r2_eq25_confidence.png']
checkpoints: ['lm_cal.pt', 'lm_cal_long.pt', 'lm_long_n1100.pt', 'lm_long_n120.pt', 'lm_long_n1500.pt', 'lm_long_n2100.pt', 'lm_long_n250.pt', 'lm_long_n3000.pt', 'lm_long_n500.pt', 'lm_long_n60.pt', 'lm_long_n800.pt', 'lm_sat.pt', 'rm_long.pt', 'rm_long_n2100.pt', 'rm_v2.pt']


In [ ]:
# ---------------------------------------------------------------------------
# Figure: Theorem 4.1. Three independent computations of the same distribution.
# ---------------------------------------------------------------------------
fig = plt.figure(figsize=(14.0, 6.6))
ext = [THM["LO"], THM["HI"], THM["LO"], THM["HI"]]
vmax = max(THM["A"].max(), THM["C"].max()) * 0.85

panels = [("$\\pi_{LLM}$  (reference policy)", THM["PI"], "the model before any reward"),
          ("A.  closed form  $\\rho^\\star \\propto \\pi e^{\\lambda r}$", THM["A"], "one line of algebra"),
          ("B.  $\\arg\\min_\\rho \\mathcal{L}_{PPO}$", THM["B"], "gradient descent over a 65k-dim simplex"),
          ("C.  Langevin particles (MALA)", THM["C"], "400k independent noisy trajectories")]
for i, (t, M, sub) in enumerate(panels):
    ax = fig.add_subplot(2, 4, i+1)
    ax.imshow(M.T[::-1], extent=ext, cmap="magma", vmin=0, vmax=vmax, aspect="auto")
    ax.set_title(t, fontsize=9)
    ax.text(0.5, -0.155, sub, transform=ax.transAxes, ha="center", fontsize=7.2, color="#555")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

# convergence
ax = fig.add_subplot(2, 3, 4)
ss = list(THM["snaps"])
ax.semilogx(ss, [THM["tvC"][s] for s in ss], "-o", ms=3.5, lw=2, color=C["dto"],
            label="from $\\pi_{LLM}$  (Thm 4.1's condition)")
ax.semilogx(ss, [THM["tvD"][s] for s in ss], "-s", ms=3.5, lw=2, color=C["accent"],
            label="from uniform (wrong start)")
ax.axhline(THM["floor"], color=C["grey"], ls="--", lw=1.3)
ax.axhline(THM["tv0"], color=C["base"], ls=":", lw=1.3)
ax.annotate(f"noise floor {THM['floor']:.3f}", (ss[0], THM["floor"]), textcoords="offset points",
            xytext=(2, 4), fontsize=7.2, color=C["grey"])
ax.annotate(f"$TV(\\pi_{{LLM}},\\rho^\\star)$ = {THM['tv0']:.3f}", (ss[0], THM["tv0"]),
            textcoords="offset points", xytext=(2, -11), fontsize=7.2, color=C["base"])
ax.set_xlabel("Langevin steps"); ax.set_ylabel(r"$TV(\cdot\,,\,\rho^\star)$")
ax.set_title("convergence to the RL optimum"); ax.legend(fontsize=7.2)

# mode masses
ax = fig.add_subplot(2, 3, 5)
xs, w = np.arange(len(MIX["mm_pi"])), 0.26
ax.bar(xs-w, MIX["mm_pi"],   w, label=r"$\pi_{LLM}$",   color=C["base"])
ax.bar(xs,   MIX["mm_star"], w, label=r"$\rho^\star$ exact", color=C["exact"])
ax.bar(xs+w, MIX["mm_C"],    w, label="Langevin",       color=C["dto"])
ax.set_xticks(xs); ax.set_xticklabels([f"mode {i}" for i in xs])
ax.set_ylabel("probability mass")
ax.set_title("reward MIGRATES mass between modes\n(not a local reshaping)")
ax.legend(fontsize=7.2)

# what the residual was
ax = fig.add_subplot(2, 3, 6)
bars = {"$\\pi_{LLM}$\n(no descent)": THM["tv0"],
        "unadjusted\nEuler-Maruyama": 0.24177,
        "MALA\n(exact)": 0.02563,
        "noise\nfloor": THM["floor"]}
cols = [C["base"], C["bad"], C["dto"], C["grey"]]
ax.bar(range(len(bars)), list(bars.values()), color=cols, width=.62)
for i, v in enumerate(bars.values()):
    ax.annotate(f"{v:.3f}", (i, v), textcoords="offset points", xytext=(0, 3),
                ha="center", fontsize=7.6)
ax.set_xticks(range(len(bars))); ax.set_xticklabels(list(bars), fontsize=7.4)
ax.set_ylabel(r"$TV(\cdot\,,\,\rho^\star)$")
ax.set_title("where the residual came from\n(discretisation, then mixing)")

fig.suptitle("Theorem 4.1: test-time gradient descent reaches the KL-regularised RL optimum",
             fontsize=11, fontweight="bold", y=1.0)
fig.tight_layout()
fig_show(fig, "r8_theorem41")

In [ ]:
# ---------------------------------------------------------------------------
# Long-world evaluation. Identical protocol to the short world, so the two are
# directly comparable. The quantity of interest is NOT either world's absolute
# accuracy, it is how the nabla-Reasoner-vs-BoN gap MOVES between them.
# ---------------------------------------------------------------------------
N_LW, SEEDS_LW = 600, 2
seed_all(0)
lw_idx = torch.randperm(len(TEST_P))[:N_LW].to(DEV)
LW     = X_TE[lw_idx]
plw    = LW[:, :PLEN]
def acc_lw(seq): return is_correct(seq, LW).float().mean().item()

def run_sc_long(prompts, N, tag="swsl", seed=0):
    """Majority vote on the FINAL answer; return a candidate that voted for the winner."""
    reset_calls()
    gen   = torch.Generator(device=DEV).manual_seed(seed)
    cands = torch.stack([rollout_with_logits(prompts, temp=TEMP, gen=gen)[0] for _ in range(N)])
    ansi, valid = answer_index(cands)                                   # (N,B)
    votes = (F.one_hot(ansi.T, 10000) * valid.T.unsqueeze(-1)).sum(1)   # (B,10000)
    win   = votes.argmax(-1)                                            # (B,)
    match = (ansi == win.unsqueeze(0)) & valid                          # (N,B)
    pick  = torch.where(match.any(0), match.float().argmax(0), torch.zeros_like(win))
    out   = cands[pick, torch.arange(len(prompts), device=DEV)]
    STATS[tag] = dict(fwd=CALLS["fwd"], bwd=0, calls=CALLS["fwd"])
    return out

def run_nr_l(lam, lr, N_max=8, seeds=SEEDS_LW):
    a, c = [], 0
    for s in range(seeds):
        a.append(acc_lw(nabla_reasoner(plw, lam, lr=lr, accel=True, N_max=N_max,
                                       seed=s, tag="lw")))
        c = STATS["lw"]["calls"]
    return float(np.mean(a)), float(np.std(a)), c

t0 = time.time()
print(f"long world: {N_LW} problems x {SEEDS_LW} seeds | greedy = {GREEDY_ACC_L*100:.2f}%\n")
print(f"lambda sweep for nabla-Reasoner (lr = 0.1, the short world's best)")
print(f"  {'lambda':>9}{'accuracy':>18}{'calls':>9}")
SWEEP_L = {}
for lam in [1.0, 5.0, 15.0, 40.0, 87.4]:
    m, sd, c = run_nr_l(lam, 0.1)
    SWEEP_L[lam] = (m, sd, c)
    print(f"  {lam:>9.3g}{m*100:>11.2f}+/-{sd*100:<5.2f}{c:>9}")
bl_l, (bm_l, bsd_l, bc_l) = max(SWEEP_L.items(), key=lambda kv: kv[1][0])
print(f"  best: lambda={bl_l:g} -> {bm_l*100:.2f}% +/- {bsd_l*100:.2f} at {bc_l} calls "
      f"({time.time()-t0:.0f}s)")

print(f"\ntest-time scaling in the LONG world")
CURVE_L = {"BoN": [], "SC": [], "nabla": []}
for N in [2, 4, 8, 16, 32, 56]:
    ab, cb = [], 0
    for s in range(SEEDS_LW):
        ab.append(acc_lw(run_bon(plw, N, tag="lwb", seed=s))); cb = STATS["lwb"]["calls"]
    CURVE_L["BoN"].append((cb, float(np.mean(ab)), float(np.std(ab))))
    asc, cs = [], 0
    for s in range(SEEDS_LW):
        asc.append(acc_lw(run_sc_long(plw, N, seed=s))); cs = STATS["swsl"]["calls"]
    CURVE_L["SC"].append((cs, float(np.mean(asc)), float(np.std(asc))))
for nmax in [1, 2, 4, 8]:
    CURVE_L["nabla"].append(run_nr_l(bl_l, 0.1, N_max=nmax))

print(f"  {'calls':>7}{'BoN':>17}{'SC':>17}   |{'calls':>7}{'nabla-Reasoner':>20}")
for i in range(6):
    cb_, mb_, sb_ = CURVE_L["BoN"][i]; cs_, ms_, ss_ = CURVE_L["SC"][i]
    pt = CURVE_L["nabla"][i] if i < len(CURVE_L["nabla"]) else None
    tail = f"   |{pt[2]:>7}{pt[0]*100:>13.2f}+/-{pt[1]*100:<4.2f}" if pt else ""
    print(f"  {cb_:>7}{mb_*100:>11.2f}+/-{sb_*100:<4.2f}{ms_*100:>11.2f}+/-{ss_*100:<4.2f}{tail}")

# --- THE COMPARISON: how many times cheaper is BoN, in each world? ----------
def cheaper_factor(curve, nr_acc, nr_calls):
    hits = [x for x in curve["BoN"] if x[0] <= nr_calls and x[1] >= nr_acc]
    if not hits: return None
    return nr_calls / min(hits, key=lambda x: x[0])[0]

f_short = cheaper_factor(SHORT["CURVE"], SHORT["BEST"]["acc"], SHORT["BEST"]["calls"])
f_long  = cheaper_factor(CURVE_L, bm_l, bc_l)
print(f"\n{'='*84}")
print(f"THE LENGTH EXPERIMENT -- how many times CHEAPER is BoN at matching nabla-Reasoner?")
print(f"  {'world':<14}{'answer len':>12}{'nabla acc':>12}{'calls':>8}{'BoN cheaper by':>18}")
print(f"  {'short':<14}{SHORT['YLEN']:>12}{SHORT['BEST']['acc']*100:>11.2f}%"
      f"{SHORT['BEST']['calls']:>8}{(f'{f_short:.1f}x' if f_short else 'never'):>18}")
print(f"  {'long':<14}{YLEN:>12}{bm_l*100:>11.2f}%{bc_l:>8}"
      f"{(f'{f_long:.1f}x' if f_long else 'never'):>18}")
if f_short and f_long:
    print(f"\n  the disadvantage {'SHRANK' if f_long < f_short else 'GREW'} by "
          f"{f_short/f_long:.2f}x when answers got {YLEN/SHORT['YLEN']:.2f}x longer")
    print(f"  >>> length hypothesis {'SUPPORTED' if f_long < f_short*0.7 else 'NOT SUPPORTED'}")
elif f_long is None:
    print(f"\n  >>> in the LONG world no cheaper BoN budget matches nabla-Reasoner.")
    print(f"      C5 holds here but not in the short world -> length hypothesis SUPPORTED.")
LENGTH_EXP = dict(f_short=f_short, f_long=f_long, curve_l=CURVE_L, best_l=(bl_l, bm_l, bsd_l, bc_l))